# TAPAS large (WTQ) — DIMER E2E table question answering fine-tuning tutorial (standalone)

[![GitHub](https://img.shields.io/badge/GitHub-181717?style=flat&logo=github&logoColor=white)](https://github.com/kurtvalcorza/tapas-table-question-answering-pipeline) [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kurtvalcorza/tapas-table-question-answering-pipeline/blob/main/tutorials/tapas_table_qa_colab.ipynb) [![Hugging Face](https://img.shields.io/badge/%F0%9F%A4%97%20Hugging%20Face-google%2Ftapas--large--finetuned--wtq-ffcc4d?style=flat)](https://huggingface.co/google/tapas-large-finetuned-wtq) [![Upstream](https://img.shields.io/badge/Upstream-google--research%2Ftapas-181717?style=flat&logo=github&logoColor=white)](https://github.com/google-research/tapas) [![arXiv](https://img.shields.io/badge/arXiv-2004.02349-b31b1b.svg)](https://arxiv.org/abs/2004.02349)

**Profile:** `E2E`  
**Mode:** `GUIDED`  
**Notebook specification:** DIMER Notebook Specification 2.0 — **standalone** (§4)  
**Capability:** table question answering (one table of string cells + one question → selected cells, one aggregation operator NONE/SUM/AVERAGE/COUNT, and a pipeline-computed numeric answer) and bounded supervised fine-tuning of the last encoder blocks and heads on a labelled table-question set, using the pinned TAPAS-large WTQ weights

**This notebook is standalone.** It carries the repository's package (3 modules under `src/tapas_table_qa_pipeline/`, at revision `25ac5488e7c2`) verbatim in Section 2, the pinned model identity and the per-file SHA-256 manifest in Section 3, and the exact runtime pins in Section 1, so it keeps working after export even if the repository changes or disappears. Its only external dependencies are the pinned Python distributions and the Hugging Face Hub at the immutable revision `f58317ab2577d17647d9acafa790c744a0388b30` (~1347 MB, digest-verified before loading). It was generated by `tools/build_notebook.py` (build_notebook.py/2); edit the repository and regenerate rather than editing cells.

**Run all:** Selecting **Run all** in a fresh supported runtime installs the pinned dependencies, stages and digest-verifies the pinned `google/tapas-large-finetuned-wtq` snapshot (a 1.35 GB `model.safetensors`; no pickle is opened anywhere), fetches the pinned WikiSQL validation shard from the Hugging Face Hub (3.6 MB, refused on any byte-size or SHA-256 mismatch), executes its SQL programmes in pure Python for the gold cells and values, draws a table-disjoint sample of 240 / 90 / 150 questions stratified by operator, answers the synthetic city table through the inference contract with an input manifest and a rejection probe, measures the frozen model's denotation, operator and cell accuracy per question type over the 150 test questions beside the first-cell and keyword-lookup baselines, runs a bounded fine-tuning of the last two encoder blocks and the three heads with the model's own weak-supervision loss and validation-score epoch selection, scores the held-out questions again per type, re-answers the city table with the adapted model, exports the adapter as safetensors with a manifest, and reloads that artifact into a fresh pipeline to verify answer parity. The default path needs no repository clone, no DIMER worker or service, no credential, no upload dialog and no configuration edit (NOTEBOOK_SPEC 2.0 §5). TAPAS-large is a 24-layer encoder over up to 512 tokens: on the build workstation's CPU the whole path took about 31 minutes after the downloads (expect a multiple of that on a 2-vCPU hosted runtime); a CUDA runtime is used automatically when present and finishes in a few minutes.

**Bring Your Own Data:** After the tutorial workflow completes, set `USE_BYOD = True` in Section 4 and re-run from that cell to upload one JSONL file of `{id, table, question, answer, category?}` records (Section 4 and the Prerequisites state the shape) — at least eight questions over at least two tables. They pass through the same validation, table-disjoint split, baselines, fine-tuning, held-out evaluation, artifact export and reload-parity cells as the WikiSQL sample. Uploaded tables stay inside this runtime. BYOD is optional and never part of the default path.

`google/tapas-large-finetuned-wtq` is the TAPAS model of Herzig et al. (2020) — a 24-layer BERT-style encoder over a flattened table (`[CLS] question [SEP] header row + data rows [SEP]`, lower-cased WordPiece with row, column and numeric-rank ids per token) with a cell-selection head and an aggregation head — fine-tuned by Google on WikiTableQuestions after SQA and WikiSQL; 336,734,214 parameters, published under the **Apache-2.0** licence. A cell is **selected** when the mean sigmoid probability over its tokens exceeds `CELL_THRESHOLD` (0.5), the operator is an `argmax` over `NONE`, `SUM`, `AVERAGE`, `COUNT`, and **the numeric answer is the pipeline's arithmetic over the selected cell strings**, never a model output. Neither decision is a calibrated probability, the model never abstains, and it applies no acceptance threshold — **every question yields some cells and some operator**.

What this notebook adds to inference is **adaptation with labelled questions**. The dataset is real and shifted from the checkpoint's own training data: **WikiSQL** (Zhong et al. 2017, **BSD-3-Clause**) pairs Wikipedia tables with questions and the SQL programme that answers them; the pipeline executes each programme in pure Python to obtain the gold cells and value, maps WikiSQL's `MAX`/`MIN` to `NONE` with the extreme cell (TAPAS-WTQ has no such operators) and its `AVG` to `AVERAGE`, and keeps the original operator as the question's `category`. The WTQ checkpoint is already a strong lookup model on these tables (the build record measured denotation accuracy 0.82 frozen on the 150 test questions, 0.92 on lookups) but chooses the right operator for only about half the questions (0.57), so the honest question is narrow: does a bounded fine-tuning of the last two encoder blocks and the three heads on 240 questions — 80 lookups and 32 per operator, the model's own **weak-supervision loss** (the gold cells label a lookup, an operator question carries only its number) — move the **denotation accuracy**, the **aggregation accuracy** and the **cell accuracy** on a table-disjoint test split, per question type, against two **non-neural baselines** (the **first-cell floor** and a **keyword lookup**)? Nothing here is a quality claim about your tables: it is one seeded split of one sample.

**Snapshot note:** the pinned revision ships `model.safetensors` (a 6-file manifest) — no pickle is opened anywhere in this notebook. Section 3 stages and digest-verifies those files before the tokenizer or the model is constructed.

**Learning objectives:** install the pinned runtime; read what the carried package guarantees; stage and digest-verify the immutable upstream snapshot; fetch a digest-pinned labelled table-question set, execute its programmes for gold answers, validate it and split it by table without leakage; answer a synthetic table through the public API and read the cell/operator/numeric contract correctly (fixed threshold, argmax, pipeline arithmetic, no abstention); measure the frozen model's denotation, operator and cell accuracy per question type beside two non-neural baselines; run a bounded fine-tuning with the model's own weak-supervision loss, explicit hyperparameters and validation-based epoch selection; evaluate on a table-disjoint test split; re-answer the synthetic table with the adapted model; and export a safetensors adapter that reloads against the pinned base with verified parity.

**This notebook does not demonstrate:** conversational or multi-turn table QA (the SQA setting), tables with more than `MAX_ROWS` rows or `MAX_COLUMNS` columns, non-English tables, text-to-SQL or arbitrary computation beyond the four aggregation operators, the `MAX`/`MIN` operators themselves (WikiSQL's become a lookup of the extreme cell), fine-tuning of the embeddings or of any encoder block but the last ones, evaluation on WikiTableQuestions or WikiSQL proper (only one seeded 480-question sample is scored here), and any claim that six question types on Wikipedia tables stand in for your tables. The repository exposes none of these.

## Prerequisites

- **Runtime:** a fresh supported runtime (Google Colab or Jupyter, Python 3.12). The default path runs on CPU (float32) and uses CUDA automatically when available (also float32). CPU is slow: TAPAS-large answers a question in about 0.85 s on the build workstation's CPU (the build record measured 76 s to score the 150 test questions and 1,401 s for the four epochs of fine-tuning over 240 questions with per-epoch validation scoring); the whole default path took 1,865 s there with the snapshot and the shard already cached, and 148 s on an RTX 5070 Ti. A 2-vCPU hosted runtime will take a multiple of the workstation figure. The pinned `torch==2.14.0` install and the 1.35 GB checkpoint are the large downloads of the run; the WikiSQL shard adds 3.6 MB.
- **Knowledge:** basic Python; what a sigmoid threshold and an argmax are and why neither is a calibrated probability; that an aggregation over selected cells is arithmetic the pipeline performs, not a model output; what denotation accuracy measures and why one seeded split gives no dispersion.
- **Data contract:** records are `{id, table, question, answer}` with an optional `category` — `table` is `{column: [cells]}` with every header and cell a str (at most `MAX_ROWS` = 64 rows, `MAX_COLUMNS` = 32 columns, `MAX_CELL_CHARS` = 200 characters per cell), `question` a non-empty str of at most `MAX_QUERY_CHARS` = 500 characters, `answer` = `{aggregation: NONE|SUM|AVERAGE|COUNT, coordinates: [[row, col], ...], denotation: [cell, ...] | number}` where a `NONE` denotation is the selected cells and an operator denotation is the number the operator produces from them (validated). Ids match `[A-Za-z0-9_.:-]{1,64}` and are unique; a dataset needs 8..20,000 records; splitting is by table (normalised content) so no table lands in two splits; a table whose gold cells fall past the `MAX_TOKENS` = 512 truncation point is refused at training time. BYOD accepts one JSONL file (one record per line) or a JSON list in that shape.
- **Validation is structural, not semantic:** every table, question and answer is checked for shape and arithmetic consistency, but nothing checks that an answer is right — a mislabelled set is fine-tuned on without complaint.
- **Privacy:** Do not upload confidential or restricted data to a hosted runtime unless you are authorized to process it there. The default path uploads nothing.
- **External access (data):** besides the model snapshot, the default path fetches one parquet file from the Hugging Face Hub dataset repository `Salesforce/wikisql` at the immutable revision `48cfb60afd0d5f9d2231ca90f76edf9f975181bc` (`default/validation/0000.parquet`, 3,630,670 bytes, SHA-256 `ed524cc7…`, pinned in the carried `samples.py` and refused on any mismatch). WikiSQL is BSD-3-Clause (Zhong, Xiong & Socher 2017); nothing is redistributed by this repository.
- **External access:** the Hugging Face Hub only, to fetch the pinned `google/tapas-large-finetuned-wtq` snapshot (~1347 MB in total) at revision `f58317ab2577…`. No GitHub access and no credentials are required; nothing is installed from this repository.

## 1. Install the pinned runtime

The dependency set is pinned exactly (the same pins as the repository's pyproject.toml at the generating revision; any `--index-url`/`--find-links` lines are passed to pip as written) and installed directly — there is no repository clone and no package install. If a pin replaces a distribution this runtime has already imported, the cell stops with a restart instruction rather than continuing with mixed versions. Look for a dictionary reporting the notebook's source revision, Python, `torch`, `transformers`, `pandas` versions, and whether CUDA is available.

In [ ]:
import importlib
import importlib.metadata
import os
import platform
import subprocess
import sys

PINS = [
    'torch==2.14.0',
    'torchvision==0.29.0',
    'torchaudio==2.11.0',
    'transformers==4.57.6',
    'tokenizers==0.22.2',
    'huggingface-hub==0.36.2',
    'safetensors==0.8.0',
    'numpy==2.5.3',
    'pandas==3.0.5',
    'pyarrow==25.0.1',
]
NOTEBOOK_SOURCE = {
    'repository': 'tapas-table-question-answering-pipeline',
    'repository_revision': '25ac5488e7c2b845aeba3fef8511bbb923b2e8f5',
    'embedded_module': 'src/tapas_table_qa_pipeline/pipeline.py',
    'embedded_modules': ['src/tapas_table_qa_pipeline/pipeline.py', 'src/tapas_table_qa_pipeline/metrics.py', 'src/tapas_table_qa_pipeline/samples.py'],
    'module_sha256': '80f3491278c8dd7ed65919c79ed2a27d603d82c6e7f89e363deac7bf179ba2ae',
    'generator': 'build_notebook.py/2',
    'notebook_spec': '2.0',
}
SKIP_INSTALL = os.environ.get('DIMER_NOTEBOOK_CI_PREINSTALLED') == '1'

def _installed_version(distribution):
    try:
        return importlib.metadata.version(distribution)
    except importlib.metadata.PackageNotFoundError:
        return None

if not SKIP_INSTALL:
    # Capture every distribution already imported in this runtime, whatever its module name
    # (PIL -> pillow), so a pinned install that replaces a loaded package is detected and the
    # notebook stops with a restart instruction instead of continuing with mixed versions.
    _module_dists = importlib.metadata.packages_distributions()
    _loaded = sorted({d for m in list(sys.modules) for d in _module_dists.get(m.partition('.')[0], ())})
    loaded = {distribution: _installed_version(distribution) for distribution in _loaded}
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *PINS], check=True)
    importlib.invalidate_caches()
    stale = []
    for distribution, before in loaded.items():
        installed = _installed_version(distribution)
        if before is not None and before != installed:
            stale.append(f'{distribution}: loaded={before}, installed={installed}')
    if stale:
        raise RuntimeError('Core dependencies changed while older modules were loaded: ' + '; '.join(stale) + '. Restart the runtime, then rerun from the top.')

import torch, transformers, pandas
print({'notebook_source': NOTEBOOK_SOURCE, 'python': platform.python_version(), 'torch': torch.__version__, 'transformers': transformers.__version__, 'pandas': pandas.__version__, 'cuda': torch.cuda.is_available()})

## 2. Pipeline code (carried verbatim from `src/tapas_table_qa_pipeline/` @ `25ac5488e7c2`)

The next 3 cell(s) **are** the repository's package, module by module in dependency order: the pinned identity constants, snapshot verification (`verify_snapshot`), staged download (`stage_missing_files`), the named operational ceilings, the public validation and evaluation helpers, and the pipeline class. The text is the modules', byte for byte, except for the rewrite rules listed in `tools/build_notebook.py` (1 rule(s), plus the removal of package-relative `from .x import` lines, whose names are already defined by the preceding cells). The repository's parity test (`tests/test_notebook_parity.py`) fails whenever these cells and the modules diverge, so what you run here is what the repository tests. Nothing in these cells runs a model yet.

**Module 1/3:** `src/tapas_table_qa_pipeline/pipeline.py`

In [ ]:
"""Table question answering over the pinned ``google/tapas-large-finetuned-wtq`` checkpoint.

Weights load only from a digest-verified local snapshot (``weights/<key>/``) or, when explicitly allowed,
from the Hugging Face Hub at the pinned revision. One task method: ``answer(table, query)`` — the model
selects table cells and one aggregation operator (NONE/SUM/AVERAGE/COUNT) exactly as the upstream
``TapasTokenizer.convert_logits_to_predictions`` does; the numeric value for SUM/AVERAGE/COUNT is then
computed *by this module* from the selected cell strings and labelled as such.

The adaptation contract (``evaluate``, ``adapt``, ``save_artifact``, ``load_artifact``, ``from_artifact``)
fine-tunes the last encoder blocks and the three heads on a labelled table-question set with the model's own
weak-supervision loss and exports the trained tensors as a digest-manifested safetensors adapter.
"""

# ruff: noqa: E501  -- adaptation-contract lines are kept at the fleet width

from __future__ import annotations

import hashlib
import json
import random
import re
import time
from collections.abc import Callable, Mapping, Sequence
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any

MODEL_ID = "google/tapas-large-finetuned-wtq"
MODEL_REVISION = "f58317ab2577d17647d9acafa790c744a0388b30"
MODEL_LICENSE = "apache-2.0"
MODEL_KEY = "tapas-large-wtq"
DEFAULT_WEIGHTS_DIR = Path.cwd() / "weights" / MODEL_KEY  # standalone rewrite (build_notebook.py): working-directory-relative
MANIFEST_NAME = "dimer-base-manifest.json"
WEIGHT_FILE = "model.safetensors"

# Ceilings. MAX_ROWS/MAX_COLUMNS are ``max_num_rows``/``max_num_columns`` in the pinned config.json;
# MAX_TOKENS is ``model_max_length`` in tokenizer_config.json and the fine-tuning sequence length (README).
# A table over MAX_ROWS or MAX_COLUMNS is rejected. A table that fits those but tokenises past MAX_TOKENS
# is TRUNCATED by the tokenizer (``drop_rows_to_fit``): cell texts are first capped at a common token
# count, then trailing rows are dropped until the flattened table fits; the result reports ``rows_kept``.
MAX_ROWS = 64
MAX_COLUMNS = 32
MAX_TOKENS = 512
MAX_QUERY_CHARS = 500
MAX_CELL_CHARS = 200
# ``aggregation_labels`` in config.json, index order; the aggregation head is an argmax over these four.
AGGREGATIONS = ("NONE", "SUM", "AVERAGE", "COUNT")
# ``cell_classification_threshold`` default in the upstream convert_logits_to_predictions: a cell is
# selected when the mean sigmoid probability over its tokens exceeds this value.
CELL_THRESHOLD = 0.5
DECISION_RULE = (
    f"cell selected when its mean token sigmoid probability > CELL_THRESHOLD={CELL_THRESHOLD}; aggregation "
    "operator = argmax over the four aggregation logits; numeric answer computed by the pipeline from the "
    "selected cells (COUNT = number of cells; SUM/AVERAGE parse each cell as a number, else None)"
)
NUMERIC_ANSWER_SOURCE = "computed by the pipeline from the selected cells, not emitted by the model"
_NUMBER = re.compile(r"^[-+]?(?:\d{1,3}(?:,\d{3})+|\d+)(?:\.\d+)?%?$")
PARAMETER_COUNT = 336_734_214
ENCODER_LAYERS = 24
DEFAULT_TRAINABLE_LAYERS = 2  # last encoder blocks; plus the cell, column and aggregation heads
HEAD_PREFIXES = ("output_weights", "output_bias", "column_output_weights", "column_output_bias", "aggregation_classifier")
ARTIFACT_FORMAT = f"org.valcorza.{MODEL_KEY}.adapter.v1"
ARTIFACT_VERSION = "1.0"
ADAPTER_WEIGHTS = "adapter.safetensors"
ADAPTER_MANIFEST = "manifest.json"
MIN_SCORED_RECORDS = 50  # below this a scored set is labelled a small sample
MAX_EVAL_RECORDS = 20_000


def _sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


def _read_manifest(root: Path) -> dict[str, Any]:
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"snapshot manifest not found: {manifest_path}")
    with open(manifest_path, encoding="utf-8") as fh:
        return json.load(fh)


def verify_snapshot(path: str | Path | None = None) -> dict[str, Any]:
    """Check a local snapshot against its manifest; raise naming the first mismatch."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest = _read_manifest(root)
    if manifest.get("modelId") != MODEL_ID:
        raise ValueError(f"manifest modelId {manifest.get('modelId')!r} != {MODEL_ID!r}")
    if manifest.get("revision") != MODEL_REVISION:
        raise ValueError(f"manifest revision {manifest.get('revision')!r} != {MODEL_REVISION!r}")
    for entry in manifest.get("files", []):
        file_path = root / entry["path"]
        if not file_path.is_file():
            raise FileNotFoundError(f"snapshot file missing: {file_path}")
        size = file_path.stat().st_size
        if size != entry["bytes"]:
            raise ValueError(f"{entry['path']}: size {size} != manifest {entry['bytes']}")
        digest = _sha256(file_path)
        if digest != entry["sha256"]:
            raise ValueError(f"{entry['path']}: sha256 {digest} != manifest {entry['sha256']}")
    return {"path": str(root), **manifest}


def _hub_download(relative_path: str, root: Path) -> None:
    """Fetch one manifest-listed file at MODEL_REVISION straight into the snapshot directory."""
    from huggingface_hub import hf_hub_download

    hf_hub_download(MODEL_ID, relative_path, revision=MODEL_REVISION, local_dir=str(root))


def stage_missing_files(
    path: str | Path | None = None,
    *,
    allow_download: bool = False,
    downloader: Callable[[str, Path], None] | None = None,
) -> list[str]:
    """Fetch manifest-listed files that are absent locally (a fresh clone commits the manifest but
    git-ignores the weights). Returns the relative paths fetched; `verify_snapshot` still runs after."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest = _read_manifest(root)
    if manifest.get("modelId") != MODEL_ID or manifest.get("revision") != MODEL_REVISION:
        raise ValueError(
            f"manifest names {manifest.get('modelId')}@{manifest.get('revision')}, "
            f"package pins {MODEL_ID}@{MODEL_REVISION}; refusing to stage"
        )
    missing = [entry["path"] for entry in manifest["files"] if not (root / entry["path"]).is_file()]
    if not missing:
        return []
    if not allow_download:
        raise FileNotFoundError(
            f"snapshot at {root} is missing {missing}; "
            f"pass allow_download=True to fetch them at {MODEL_REVISION}"
        )
    fetch = downloader or _hub_download
    for relative_path in missing:
        fetch(relative_path, root)
    return missing


def _check_cell(value: Any, where: str) -> str:
    if not isinstance(value, str):
        raise TypeError(f"{where} must be str (stringify numbers yourself), got {type(value).__name__}")
    if len(value) > MAX_CELL_CHARS:
        raise ValueError(f"{where} has {len(value)} chars; ceiling is MAX_CELL_CHARS={MAX_CELL_CHARS}")
    return value


def _check_table(table: Any) -> tuple[list[str], list[list[str]]]:
    """Normalise ``{column: [cells]}`` or ``[{column: cell}, ...]`` to (columns, rows); raise on the
    first violated ceiling. Every header and cell must already be a str: TAPAS tokenises text only."""
    if isinstance(table, Mapping):
        columns = [_check_cell(name, f"column name {name!r}") for name in table]
        for name, col in table.items():
            if isinstance(col, str | bytes) or not isinstance(col, Sequence):
                raise TypeError(f"column {name!r} must be a list of str cells, got {type(col).__name__}")
        lengths = {len(col) for col in table.values()}
        if len(columns) and len(lengths) != 1:
            raise ValueError(
                f"columns have unequal lengths {sorted(lengths)}; every column needs one cell per row"
            )
        n_rows = lengths.pop() if lengths else 0
        rows = [[table[name][i] for name in columns] for i in range(n_rows)]
    elif isinstance(table, Sequence) and not isinstance(table, str | bytes):
        if not table or not all(isinstance(row, Mapping) for row in table):
            raise TypeError(
                "table must be a non-empty {column: [cells]} mapping or a list of {column: cell} rows"
            )
        columns = [_check_cell(name, f"column name {name!r}") for name in table[0]]
        rows = []
        for i, row in enumerate(table):
            if list(row) != columns:
                raise ValueError(f"row {i} has columns {list(row)}; every row must carry exactly {columns}")
            rows.append([row[name] for name in columns])
    else:
        raise TypeError("table must be a {column: [cells]} mapping or a list of {column: cell} rows")
    if not 1 <= len(columns) <= MAX_COLUMNS:
        raise ValueError(f"table has {len(columns)} columns; ceiling is 1..MAX_COLUMNS={MAX_COLUMNS}")
    if not 1 <= len(rows) <= MAX_ROWS:
        raise ValueError(f"table has {len(rows)} rows; ceiling is 1..MAX_ROWS={MAX_ROWS}")
    checked = [[_check_cell(v, f"cell[{r}][{c}]") for c, v in enumerate(row)] for r, row in enumerate(rows)]
    return columns, checked


def _check_query(query: Any) -> str:
    if not isinstance(query, str):
        raise TypeError(f"query must be str, got {type(query).__name__}")
    if not query.strip():
        raise ValueError("query is empty")
    if len(query) > MAX_QUERY_CHARS:
        raise ValueError(f"query has {len(query)} chars; ceiling is MAX_QUERY_CHARS={MAX_QUERY_CHARS}")
    return query


def parse_number(cell: str) -> float | None:
    """Parse one cell string as a number (optional sign, thousands commas, decimals, trailing %) or None."""
    text = cell.strip()
    if not _NUMBER.match(text):
        return None
    return float(text.rstrip("%").replace(",", ""))


def compute_numeric_answer(cells: Sequence[str], aggregation: str) -> tuple[float | None, list[str]]:
    """The pipeline-computed value for an aggregation: (value, cells that did not parse as numbers)."""
    if aggregation == "COUNT":
        return float(len(cells)), []
    if aggregation not in ("SUM", "AVERAGE") or not cells:
        return None, []
    parsed = [(cell, parse_number(cell)) for cell in cells]
    unparsed = [cell for cell, value in parsed if value is None]
    if unparsed:
        return None, unparsed
    values = [value for _, value in parsed if value is not None]
    return (sum(values) if aggregation == "SUM" else sum(values) / len(values)), []


def _normalise(text: str) -> str:
    return " ".join(text.lower().split())


def denotation_match(result: Mapping[str, Any], gold: str | float | int | Sequence[str]) -> bool:
    """WTQ-style denotation match: a numeric gold is compared with the pipeline's numeric answer (or the
    single selected cell parsed as a number) to 1e-6; otherwise the selected cells and the gold strings
    must be the same multiset after lower-casing and whitespace collapse."""
    cells = [str(c) for c in result.get("cells", [])]
    if isinstance(gold, bool):
        raise TypeError("gold must be a number, a string or a sequence of strings")
    if isinstance(gold, int | float):
        gold_number: float | None = float(gold)
    else:
        gold_number = parse_number(gold) if isinstance(gold, str) else None
    predicted = result.get("numeric_answer")
    if predicted is None and len(cells) == 1 and result.get("aggregation") == "NONE":
        predicted = parse_number(cells[0])
    if gold_number is not None and predicted is not None:
        return abs(float(predicted) - gold_number) <= 1e-6
    gold_cells = [gold] if isinstance(gold, str) else [] if isinstance(gold, int | float) else list(gold)
    return sorted(_normalise(c) for c in cells) == sorted(_normalise(str(g)) for g in gold_cells)


def denotation_accuracy(results: Sequence[Mapping[str, Any]], golds: Sequence[Any]) -> float:
    """Fraction of (result, gold) pairs whose denotations match; raises when the lengths differ."""
    if len(results) != len(golds) or not results:
        raise ValueError("results and golds must be non-empty and the same length")
    return sum(denotation_match(r, g) for r, g in zip(results, golds, strict=True)) / len(results)


INPUT_SCHEMA: dict[str, Any] = {
    "input": (
        "one table as {column: [cells]} or [{column: cell}, ...] with every header and cell a str, plus one "
        "non-empty question str; the model answers one question per call"
    ),
    "rows": [1, MAX_ROWS],
    "columns": [1, MAX_COLUMNS],
    "tokens": [1, MAX_TOKENS],
    "query_chars": [1, MAX_QUERY_CHARS],
    "cell_chars": [0, MAX_CELL_CHARS],
    "aggregations": list(AGGREGATIONS),
    "cell_threshold": CELL_THRESHOLD,
    "decision_rule": DECISION_RULE,
    "preprocessing": (
        "the table is flattened to [CLS] question [SEP] header row + data rows [SEP], lower-cased WordPiece; "
        "a flattened sequence over MAX_TOKENS is truncated by drop_rows_to_fit (cell texts capped at a "
        "common token count, then trailing rows dropped) and the result reports truncated, rows_kept and "
        "tokens_before_truncation; cells stay "
        "strings for the tokenizer and are parsed back to numbers only for the pipeline-computed "
        "SUM/AVERAGE answer"
    ),
}


def validate_inputs(
    table: Mapping[str, Sequence[str]] | Sequence[Mapping[str, str]],
    queries: Sequence[str],
    *,
    names: Sequence[str] | None = None,
) -> dict[str, Any]:
    """Validation stage: return the input manifest (schema, table and per-query observations, verdict).

    Rejection raises exactly as ``answer`` would: both route through ``_check_table`` and ``_check_query``.
    ``answer`` takes one query per call, so ``queries`` is the batch the notebook loops over against the
    same table. The token ceiling (``MAX_TOKENS``) needs the loaded tokenizer, so it is not observable here;
    ``answer`` reports ``rows_kept`` and ``truncated`` after tokenisation.
    """
    columns, rows = _check_table(table)
    if isinstance(queries, str | bytes) or not isinstance(queries, Sequence):
        raise TypeError("queries must be a sequence of str, not a single string")
    if not queries:
        raise ValueError("queries must hold at least one item")
    checked = [_check_query(q) for q in queries]
    if names is not None and len(names) != len(checked):
        raise ValueError("names must have one entry per query")
    return {
        "schema": dict(INPUT_SCHEMA),
        "table": {
            "columns": columns,
            "n_rows": len(rows),
            "n_columns": len(columns),
            "numeric_cells": sum(parse_number(cell) is not None for row in rows for cell in row),
        },
        "inputs": [
            {"id": names[i] if names else f"query-{i}", "chars": len(q), "query": q}
            for i, q in enumerate(checked)
        ],
        "verdict": "accepted",
        "findings": [],
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }


def evaluation_report(
    results: Mapping[str, Any] | Sequence[Mapping[str, Any]],
    golds: Sequence[Any] | None = None,
    *,
    sample_kind: str = "synthetic",
) -> dict[str, Any]:
    """Evaluation stage: a machine-readable report even when nothing is measurable.

    With ``golds`` (one gold denotation per result: a number, a string or a list of cell strings) the report
    carries ``denotation_accuracy`` as sample-sanity evidence; without them the verdict is ``not-measurable``
    and the report says what labelled data would make the task measurable.
    """
    items = [results] if isinstance(results, Mapping) else list(results)
    base = {
        "task": "table question answering (cell selection + aggregation, WikiTableQuestions fine-tune)",
        "decision_rule": DECISION_RULE,
        "sample_kind": sample_kind,
        "n_results": len(items),
        "aggregations": [r.get("aggregation") for r in items],
        "baselines": [],
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }
    if golds is None:
        return {
            **base,
            "metrics": [],
            "verdict": "not-measurable",
            "reason": "no gold denotation was supplied for the evaluated questions",
            "needs": (
                "one gold denotation per question (the cell strings, or the number an aggregation should "
                "produce) over enough questions from the deployment's own tables to state a dispersion, "
                "scored with denotation_accuracy; the upstream WTQ dev figure is not reproduced here"
            ),
        }
    return {
        **base,
        "metrics": [
            {
                "id": "denotation_accuracy",
                "value": denotation_accuracy(items, list(golds)),
                "n": len(items),
                "estimation": "single sample, no dispersion estimate",
            }
        ],
        "verdict": "sample-sanity",
        "reason": (
            f"{len(items)} question(s) with author-supplied gold denotations on one sample table; not a "
            "benchmark"
        ),
        "needs": (
            "a labelled table-question set from the deployment domain for any generalisable accuracy claim"
        ),
    }


def _trainable_names(model: Any, trainable_layers: int) -> list[str]:
    """The last `trainable_layers` encoder blocks plus the cell-selection, column and aggregation heads.
    Embeddings, the pooler and every earlier block stay frozen."""
    if isinstance(trainable_layers, bool) or not isinstance(trainable_layers, int) or not 0 <= trainable_layers <= ENCODER_LAYERS:
        raise ValueError(f"trainable_layers must be an int in 0..{ENCODER_LAYERS}")
    n_layers = len(model.tapas.encoder.layer)
    blocks = tuple(f"tapas.encoder.layer.{i}." for i in range(n_layers - trainable_layers, n_layers))
    return [name for name, _ in model.named_parameters() if name.startswith(blocks) or name.startswith(HEAD_PREFIXES)]


def _check_artifact_manifest(manifest: Mapping[str, Any], artifact_dir: Path, base_sha256: str) -> None:
    """Refuse an adapter that names another base, another format or a file that does not match its digest."""
    if manifest.get("format") != ARTIFACT_FORMAT:
        raise ValueError(f"artifact format {manifest.get('format')!r} != {ARTIFACT_FORMAT!r}")
    base = manifest.get("base", {})
    if base.get("model_id") != MODEL_ID or base.get("revision") != MODEL_REVISION:
        raise ValueError(f"artifact was trained on {base.get('model_id')}@{base.get('revision')}, not {MODEL_ID}@{MODEL_REVISION}")
    if base.get("weight_sha256") != base_sha256:
        raise ValueError("artifact base weight digest does not match the verified snapshot")
    files = manifest.get("files") or []
    if len(files) != 1 or files[0].get("path") != ADAPTER_WEIGHTS:
        raise ValueError(f"artifact manifest must list exactly {ADAPTER_WEIGHTS}")
    weights = artifact_dir / ADAPTER_WEIGHTS
    if not weights.is_file():
        raise FileNotFoundError(f"artifact weights missing: {weights}")
    size = weights.stat().st_size
    if size != files[0].get("bytes"):
        raise ValueError(f"{ADAPTER_WEIGHTS}: size {size} != manifest {files[0].get('bytes')}")
    digest = _sha256(weights)
    if digest != files[0].get("sha256"):
        raise ValueError(f"{ADAPTER_WEIGHTS}: sha256 {digest} != manifest {files[0].get('sha256')}")
    adapter = manifest.get("adapter") or {}
    names = manifest.get("tensors") or []
    if not names or any(not str(n).startswith(("tapas.encoder.layer.", *HEAD_PREFIXES)) for n in names):
        raise ValueError("artifact tensors must all belong to the encoder blocks or the heads")
    layers = adapter.get("trainable_layers")
    if isinstance(layers, bool) or not isinstance(layers, int) or not 0 <= layers <= ENCODER_LAYERS:
        raise ValueError("artifact adapter.trainable_layers must be an int in 0..ENCODER_LAYERS")


@dataclass
class TAPASTableQAPipeline:
    """``_runner(columns, rows, query)`` -> ``{coordinates: [(row, col)], aggregation_index,
    aggregation_logits, n_tokens, full_tokens, rows_kept}`` where coordinates index data rows (0 = first row
    below the header) and full_tokens is the untrimmed length; injectable so tests run offline."""

    _runner: Callable[[list[str], list[list[str]], str], Mapping[str, Any]]
    device: str = "cpu"
    source: str = "injected"
    _model: Any = field(default=None, repr=False)
    _tokenizer: Any = field(default=None, repr=False)
    _frame_cls: Any = field(default=None, repr=False)
    weight_sha256: str | None = None
    adapter: dict[str, Any] | None = None

    @classmethod
    def from_pretrained(
        cls,
        device: str | None = None,
        weights_dir: str | Path | None = None,
        allow_download: bool = False,
    ) -> TAPASTableQAPipeline:
        root = Path(weights_dir) if weights_dir is not None else DEFAULT_WEIGHTS_DIR
        weight_sha256 = None
        if (root / MANIFEST_NAME).is_file():
            stage_missing_files(root, allow_download=allow_download)
            manifest = verify_snapshot(root)
            weight_sha256 = next((e["sha256"] for e in manifest.get("files", []) if e["path"] == WEIGHT_FILE), None)
            location, kwargs, source = str(root), dict(local_files_only=True), "local-snapshot"
        elif allow_download:
            location, kwargs, source = MODEL_ID, dict(revision=MODEL_REVISION), "hf-hub"
        else:
            raise FileNotFoundError(f"no verified snapshot at {root} and allow_download=False")
        # Refuse invalid snapshots before importing model libraries.
        import torch
        from transformers import TapasForQuestionAnswering, TapasTokenizer

        resolved_device = device or ("cuda:0" if torch.cuda.is_available() else "cpu")
        import pandas as pd  # TapasTokenizer takes a DataFrame; imported after the refusal like the others
        tokenizer = TapasTokenizer.from_pretrained(location, trust_remote_code=False, **kwargs)
        model = TapasForQuestionAnswering.from_pretrained(
            location, dtype=torch.float32, trust_remote_code=False, **kwargs
        )
        model = model.to(resolved_device).eval()
        for param in model.parameters():
            param.requires_grad_(False)

        class _PositionalRows(pd.DataFrame):
            """transformers 4.57.6's TapasTokenizer reads each ``iterrows()`` row by integer position
            (``row[col_index]``), which pandas 3 no longer accepts on a str-labelled Series; rows are yielded
            re-indexed by position so the tokenizer's numeric-value annotation runs unchanged."""

            @property
            def _constructor(self):
                return _PositionalRows

            def iterrows(self):
                for index, row in super().iterrows():
                    yield index, row.reset_index(drop=True)

        def runner(columns: list[str], rows: list[list[str]], query: str) -> dict[str, Any]:
            # dtype=object: pandas 3's default pyarrow str dtype cannot hold the tokenizer's Cell objects
            frame = _PositionalRows(rows, columns=columns, dtype=object)
            encoded = tokenizer(
                table=frame, queries=[query], truncation="drop_rows_to_fit", max_length=MAX_TOKENS,
                padding="max_length", return_tensors="pt",
            )
            with torch.inference_mode():
                out = model(**{k: v.to(resolved_device) for k, v in encoded.items()})
            coordinates, agg = tokenizer.convert_logits_to_predictions(
                encoded,
                out.logits.cpu(),
                out.logits_aggregation.cpu(),
                cell_classification_threshold=CELL_THRESHOLD,
            )
            # Untrimmed length ([CLS] query [SEP] header + cells): when it exceeds n_tokens the tokenizer
            # trimmed cell text and/or dropped rows to fit MAX_TOKENS.
            flat = [query, *columns, *(cell for row in rows for cell in row)]
            return {
                "coordinates": [(int(r), int(c)) for r, c in coordinates[0]] if coordinates else [],
                "aggregation_index": int(agg[0]),
                "aggregation_logits": out.logits_aggregation[0].float().cpu().tolist(),
                "n_tokens": int(encoded["attention_mask"].sum()),
                "full_tokens": 2 + sum(len(tokenizer.tokenize(text)) for text in flat),
                "rows_kept": int(encoded["token_type_ids"][0, :, 2].max()),
            }

        return cls(runner, resolved_device, source, model, tokenizer, _PositionalRows, weight_sha256)

    def answer(
        self, table: Mapping[str, Sequence[str]] | Sequence[Mapping[str, str]], query: str
    ) -> dict[str, Any]:
        """Select cells and an aggregation operator for one question; compute the numeric answer from them."""
        columns, rows = _check_table(table)
        query = _check_query(query)
        raw = self._runner(columns, rows, query)
        coordinates = [(int(r), int(c)) for r, c in raw["coordinates"]]
        if any(not (0 <= r < len(rows) and 0 <= c < len(columns)) for r, c in coordinates):
            raise RuntimeError(f"backend returned coordinates outside the {len(rows)}x{len(columns)} table")
        aggregation = AGGREGATIONS[int(raw["aggregation_index"])]
        cells = [rows[r][c] for r, c in coordinates]
        value, unparsed = compute_numeric_answer(cells, aggregation)
        rows_kept = int(raw.get("rows_kept", len(rows)))
        n_tokens = int(raw.get("n_tokens", 0))
        full_tokens = int(raw.get("full_tokens", n_tokens))
        return {
            "cells": cells,
            "coordinates": [list(c) for c in coordinates],
            "aggregation": aggregation,
            "answer": (f"{aggregation} > " if aggregation != "NONE" else "") + ", ".join(cells),
            "numeric_answer": value,
            "numeric_answer_source": NUMERIC_ANSWER_SOURCE,
            "unparsed_cells": unparsed,
            "aggregation_logits": dict(zip(AGGREGATIONS, map(float, raw["aggregation_logits"]), strict=True)),
            "n_tokens": n_tokens,
            "tokens_before_truncation": full_tokens,
            "rows_kept": rows_kept,
            "truncated": rows_kept < len(rows) or n_tokens < full_tokens,
            "decision_rule": DECISION_RULE,
            "device": self.device,
            "source": self.source,
            "model_id": MODEL_ID,
            "model_revision": MODEL_REVISION,
        }

    def _require_model(self) -> tuple[Any, Any, Any]:
        if self._model is None or self._tokenizer is None or self._frame_cls is None:
            raise RuntimeError("this pipeline has no loaded model (injected runner); use from_pretrained for evaluate/adapt")
        return self._model, self._tokenizer, self._frame_cls


    def _encode_for_training(self, record: Mapping[str, Any]) -> tuple[dict[str, Any], float]:
        """Tokenise one record with its gold coordinates (labels, numeric values, scales) and the float answer the
        weak-supervision loss trains the operator on (NaN for a NONE answer). Raises when the table would be
        truncated past a gold row."""

        _model, tokenizer, frame_cls = self._require_model()
        columns = list(record["table"])
        rows = [[record["table"][h][i] for h in columns] for i in range(len(record["table"][columns[0]]))]
        answer = record["answer"]
        coords = [(int(r), int(c)) for r, c in answer["coordinates"]]
        frame = frame_cls(rows, columns=columns, dtype=object)
        # Weak supervision as the WTQ checkpoint was trained: a NONE answer labels its cells; an operator answer
        # carries only its number (no cell labels), so the model's aggregate mask is 1 and the regression loss
        # trains the operator and the soft cell selection together. Labelling the cells of an operator question
        # too would let `_calculate_aggregate_mask` re-route it to cell selection whenever the frozen model
        # prefers NONE, which is exactly the failure being adapted away.
        supervised = coords if answer["aggregation"] == "NONE" else []
        encoded = tokenizer(
            table=frame,
            queries=[record["question"]],
            answer_coordinates=[supervised],
            answer_text=[[rows[r][c] for r, c in coords] if supervised else [str(answer["denotation"])]],
            truncation="drop_rows_to_fit",
            max_length=MAX_TOKENS,
            padding=False,
            return_tensors="pt",
        )
        rows_kept = int(encoded["token_type_ids"][0, :, 2].max())
        if max(r for r, _ in coords) >= rows_kept:
            raise ValueError(f"record {record['id']!r}: a gold cell lies in a row the tokenizer dropped to fit MAX_TOKENS")
        if supervised and int(encoded["labels"].sum()) == 0:
            raise ValueError(f"record {record['id']!r}: no token carries a cell label after tokenisation")
        float_answer = float("nan") if answer["aggregation"] == "NONE" else float(answer["denotation"])
        return {k: v for k, v in encoded.items()}, float_answer


    def _collate(self, records: Sequence[Mapping[str, Any]], device: Any) -> dict[str, Any]:
        import torch

        encoded = [self._encode_for_training(r) for r in records]
        length = max(e["input_ids"].shape[1] for e, _ in encoded)
        pad = {"input_ids": 0, "attention_mask": 0, "labels": 0, "numeric_values": float("nan"), "numeric_values_scale": 1.0}
        batch: dict[str, list[Any]] = {k: [] for k in (*pad, "token_type_ids")}
        for e, _ in encoded:
            extra = length - e["input_ids"].shape[1]
            for key, value in pad.items():
                batch[key].append(torch.nn.functional.pad(e[key], (0, extra), value=value))
            batch["token_type_ids"].append(torch.nn.functional.pad(e["token_type_ids"], (0, 0, 0, extra), value=0))
        out = {k: torch.cat(v).to(device) for k, v in batch.items()}
        out["float_answer"] = torch.tensor([fa for _, fa in encoded], dtype=torch.float32, device=device)
        return out


    def evaluate(self, records: Sequence[Mapping[str, Any]], *, progress: Callable[[int, int], None] | None = None) -> dict[str, Any]:
        """Answer every validated record through `answer` and score the results with `metrics.denotation_metrics`
        (denotation, aggregation and cell accuracy, per category and per gold operator)."""
        pass  # standalone rewrite (build_notebook.py): `from .metrics import denotation_metrics` removed — names are kernel globals defined by the carried modules
        pass  # standalone rewrite (build_notebook.py): `from .samples import validate_dataset` removed — names are kernel globals defined by the carried modules

        checked = validate_dataset(records, min_records=1, max_records=MAX_EVAL_RECORDS)["records"]
        started = time.perf_counter()
        results = []
        for i, record in enumerate(checked):
            results.append(self.answer(record["table"], record["question"]))
            if progress is not None:
                progress(i + 1, len(checked))
        metrics = denotation_metrics(results, checked)
        metrics.update(
            {
                "verdict": "measured" if len(checked) >= MIN_SCORED_RECORDS else "measured-small-sample",
                "adapted": self.adapter is not None,
                "truncated": sum(bool(r["truncated"]) for r in results),
                "seconds": round(time.perf_counter() - started, 3),
                "decision_rule": DECISION_RULE,
                "model_id": MODEL_ID,
                "model_revision": MODEL_REVISION,
            }
        )
        return metrics


    def adapt(
        self,
        train: Sequence[Mapping[str, Any]],
        val: Sequence[Mapping[str, Any]] | None = None,
        *,
        epochs: int = 4,
        lr: float = 5e-5,
        batch_size: int = 8,
        trainable_layers: int = DEFAULT_TRAINABLE_LAYERS,
        seed: int = 0,
        progress: Callable[[Mapping[str, Any]], None] | None = None,
    ) -> dict[str, Any]:
        """Bounded fine-tuning with the model's own weak-supervision loss: the gold cells label the tokens, the
        denotation of an operator question is the `float_answer` the aggregation is trained against. Only the
        last `trainable_layers` encoder blocks and the three heads receive gradients; AdamW (weight decay 0.01),
        gradient clipping at 1.0, seeded shuffling, no scheduler. Epoch 0 records the frozen model's validation
        metrics; the epoch with the highest validation score — the mean of denotation, aggregation and cell
        accuracy, a steadier selector than denotation accuracy alone on a small split — is kept (the final one
        without a validation split). On any exception the frozen weights are restored."""
        import torch

        pass  # standalone rewrite (build_notebook.py): `from .samples import validate_dataset` removed — names are kernel globals defined by the carried modules

        model, _tokenizer, _frame_cls = self._require_model()
        if isinstance(epochs, bool) or not isinstance(epochs, int) or not 1 <= epochs <= 50:
            raise ValueError("epochs must be an int in 1..50")
        if not isinstance(lr, int | float) or not 0.0 < float(lr) <= 1e-2:
            raise ValueError("lr must be in (0, 1e-2]")
        if isinstance(batch_size, bool) or not isinstance(batch_size, int) or not 1 <= batch_size <= 64:
            raise ValueError("batch_size must be an int in 1..64")
        train_checked = validate_dataset(train)["records"]
        val_checked = validate_dataset(val, min_records=1)["records"] if val is not None else None
        names = _trainable_names(model, trainable_layers)
        if not names:
            raise ValueError("nothing to train: trainable_layers=0 selects no encoder block")
        device = torch.device(self.device)
        name_set = set(names)
        frozen_state = {k: v.detach().clone() for k, v in model.state_dict().items() if k in name_set}
        previous_adapter = self.adapter
        history: list[dict[str, Any]] = []
        started = time.perf_counter()

        def _val(epoch: int) -> dict[str, Any] | None:
            if val_checked is None:
                return None
            result = self.evaluate(val_checked)
            out = {k: result[k] for k in ("accuracy", "aggregation_accuracy", "cell_accuracy", "n")}
            out["score"] = (out["accuracy"] + out["aggregation_accuracy"] + out["cell_accuracy"]) / 3
            return out

        try:
            for param in model.parameters():
                param.requires_grad_(False)
            params = []
            for name, param in model.named_parameters():
                if name in name_set:
                    param.requires_grad_(True)
                    params.append(param)
            n_trainable = sum(p.numel() for p in params)
            entry = {"epoch": 0, "train_loss": None, "val": _val(0), "note": "frozen model"}
            history.append(entry)
            if progress is not None:
                progress(entry)
            best_epoch, best_score = 0, (history[0]["val"] or {}).get("score", -1.0)
            best_state = frozen_state
            optimizer = torch.optim.AdamW(params, lr=float(lr), weight_decay=0.01)
            rng = random.Random(seed)
            torch.manual_seed(seed)
            for epoch in range(1, epochs + 1):
                model.train()
                order = list(train_checked)
                rng.shuffle(order)
                losses = []
                for start in range(0, len(order), batch_size):
                    batch = self._collate(order[start : start + batch_size], device)
                    output = model(**batch)
                    optimizer.zero_grad(set_to_none=True)
                    output.loss.backward()
                    torch.nn.utils.clip_grad_norm_(params, 1.0)
                    optimizer.step()
                    losses.append(float(output.loss.detach()))
                model.eval()
                entry = {"epoch": epoch, "train_loss": sum(losses) / len(losses), "val": _val(epoch)}
                history.append(entry)
                if progress is not None:
                    progress(entry)
                if val_checked is None or entry["val"]["score"] > best_score:
                    best_epoch, best_score = epoch, (entry["val"] or {}).get("score", -1.0)
                    best_state = {k: v.detach().clone() for k, v in model.state_dict().items() if k in name_set}
            model.load_state_dict(best_state, strict=False)
            for param in model.parameters():
                param.requires_grad_(False)
            model.eval()
        except BaseException:
            model.load_state_dict(frozen_state, strict=False)
            for param in model.parameters():
                param.requires_grad_(False)
            model.eval()
            self.adapter = previous_adapter
            raise
        self.adapter = {
            "trainable_layers": trainable_layers,
            "trainable_names": names,
            "n_trainable": n_trainable,
            "n_total": sum(p.numel() for p in model.parameters()),
            "epochs": epochs,
            "best_epoch": best_epoch,
            "selection": "highest validation score (mean of denotation, aggregation and cell accuracy)" if val_checked is not None else "final epoch (no validation split)",
            "lr": float(lr),
            "batch_size": batch_size,
            "seed": seed,
            "n_train": len(train_checked),
            "n_val": len(val_checked) if val_checked is not None else 0,
            "history": history,
            "seconds": round(time.perf_counter() - started, 3),
        }
        return dict(self.adapter)


    def save_artifact(self, output_dir: str | Path, metadata: Mapping[str, Any] | None = None) -> Path:
        """Write the trained tensors as safetensors plus a manifest naming the base, the digests and the training
        configuration. Requires a prior `adapt`."""
        import torch
        from safetensors.torch import save_file

        model, _tokenizer, _frame_cls = self._require_model()
        if self.adapter is None:
            raise RuntimeError("nothing to save: call adapt() first")
        out = Path(output_dir)
        out.mkdir(parents=True, exist_ok=True)
        names = list(self.adapter["trainable_names"])
        state = model.state_dict()
        tensors = {name: state[name].detach().cpu().contiguous() for name in names}
        weights = out / ADAPTER_WEIGHTS
        save_file(tensors, str(weights), metadata={"format": "pt"})
        manifest = {
            "format": ARTIFACT_FORMAT,
            "version": ARTIFACT_VERSION,
            "base": {"model_id": MODEL_ID, "revision": MODEL_REVISION, "weight_file": WEIGHT_FILE, "weight_sha256": self.weight_sha256},
            "adapter": {k: v for k, v in self.adapter.items() if k not in ("history", "trainable_names")},
            "history": self.adapter["history"],
            "tensors": names,
            "files": [{"path": ADAPTER_WEIGHTS, "bytes": weights.stat().st_size, "sha256": _sha256(weights)}],
            "torch": torch.__version__,
            "metadata": dict(metadata or {}),
        }
        with open(out / ADAPTER_MANIFEST, "w", encoding="utf-8") as handle:
            json.dump(manifest, handle, indent=2, ensure_ascii=False)
        return out


    def load_artifact(self, artifact_dir: str | Path) -> dict[str, Any]:
        """Overlay a saved adapter onto this (freshly loaded) pipeline after checking its manifest, digest and exact
        tensor set. Refuses tensors outside the encoder blocks and heads."""
        from safetensors.torch import load_file

        model, _tokenizer, _frame_cls = self._require_model()
        artifact = Path(artifact_dir)
        manifest_path = artifact / ADAPTER_MANIFEST
        if not manifest_path.is_file():
            raise FileNotFoundError(f"artifact manifest missing: {manifest_path}")
        with open(manifest_path, encoding="utf-8") as handle:
            manifest = json.load(handle)
        _check_artifact_manifest(manifest, artifact, self.weight_sha256 or "")
        expected = _trainable_names(model, int(manifest["adapter"]["trainable_layers"]))
        if sorted(manifest["tensors"]) != sorted(expected):
            raise ValueError("artifact tensor set does not match its recorded configuration")
        tensors = load_file(str(artifact / ADAPTER_WEIGHTS))
        if sorted(tensors) != sorted(expected):
            raise ValueError("artifact tensor names differ from the manifest")
        state = model.state_dict()
        for name, tensor in tensors.items():
            if tuple(tensor.shape) != tuple(state[name].shape):
                raise ValueError(f"artifact tensor {name} has shape {tuple(tensor.shape)}, base has {tuple(state[name].shape)}")
        model.load_state_dict({k: v.to(state[k].device, state[k].dtype) for k, v in tensors.items()}, strict=False)
        model.eval()
        self.adapter = {**manifest["adapter"], "trainable_names": expected, "history": manifest.get("history", [])}
        return dict(self.adapter)


    @classmethod
    def from_artifact(
        cls,
        artifact_dir: str | Path,
        *,
        device: str | None = None,
        weights_dir: str | Path | None = None,
        allow_download: bool = False,
    ) -> TAPASTableQAPipeline:
        """Load the verified base snapshot, then overlay the adapter (verified before deserialising)."""
        pipe = cls.from_pretrained(device=device, weights_dir=weights_dir, allow_download=allow_download)
        pipe.load_artifact(artifact_dir)
        return pipe

**Module 2/3:** `src/tapas_table_qa_pipeline/metrics.py` (carried verbatim; see the note above)

In [ ]:
"""Corpus-level table-QA measures and two non-neural baselines, in plain Python.

``denotation_metrics`` scores pipeline results against records of the dataset contract (``samples.py``):
denotation accuracy (WTQ-style match of the pipeline's answer against the record's denotation), aggregation
accuracy (the predicted operator equals the record's), cell accuracy (the selected coordinates equal the
record's as a set), each overall, per ``category`` and per gold aggregation. The baselines answer through the
same result shape so they are scored by the same function.
"""

# ruff: noqa: E501  -- adaptation-contract lines are kept at the fleet width

from __future__ import annotations

import re
from collections.abc import Mapping, Sequence
from typing import Any

# standalone rewrite (build_notebook.py): `from .pipeline import AGGREGATIONS, DECISION_RULE, compute_numeric_answer, denotation_match` removed — names are kernel globals defined by the carried modules

METRIC_DEFINITIONS = {
    "accuracy": (
        "fraction of questions whose pipeline answer matches the record's denotation (numeric to 1e-6 for "
        "an operator, otherwise the same multiset of cell strings after lower-casing); in 0..1"
    ),
    "aggregation_accuracy": "fraction of questions whose predicted operator equals the record's; in 0..1",
    "cell_accuracy": "fraction of questions whose selected coordinates equal the record's as a set; in 0..1",
}
_WORD = re.compile(r"[a-z0-9]+")


def _words(text: str) -> set[str]:
    return set(_WORD.findall(str(text).lower()))


def _result_matches(result: Mapping[str, Any], record: Mapping[str, Any]) -> tuple[bool, bool, bool]:
    answer = record["answer"]
    gold = answer["denotation"]
    denotation = denotation_match(result, gold if not isinstance(gold, list) else list(gold))
    aggregation = result.get("aggregation") == answer["aggregation"]
    predicted = sorted(tuple(int(v) for v in c) for c in result.get("coordinates", []))
    cells = predicted == sorted(tuple(int(v) for v in c) for c in answer["coordinates"])
    return bool(denotation), bool(aggregation), bool(cells)


def denotation_metrics(
    results: Sequence[Mapping[str, Any]], records: Sequence[Mapping[str, Any]]
) -> dict[str, Any]:
    """Score one result per record; see METRIC_DEFINITIONS. Raises when the lengths differ or nothing is scored."""
    if len(results) != len(records) or not results:
        raise ValueError("results and records must be non-empty and the same length")
    rows = []
    for result, record in zip(results, records, strict=True):
        d, a, c = _result_matches(result, record)
        rows.append(
            {
                "id": record["id"],
                "category": record.get("category"),
                "gold_aggregation": record["answer"]["aggregation"],
                "aggregation": result.get("aggregation"),
                "match": d,
                "aggregation_match": a,
                "cell_match": c,
            }
        )

    def _summary(items: Sequence[Mapping[str, Any]]) -> dict[str, float | int]:
        n = len(items)
        return {
            "n": n,
            "accuracy": sum(r["match"] for r in items) / n,
            "aggregation_accuracy": sum(r["aggregation_match"] for r in items) / n,
            "cell_accuracy": sum(r["cell_match"] for r in items) / n,
        }

    categories = sorted({r["category"] for r in rows if r["category"] is not None})
    return {
        **_summary(rows),
        "per_category": {c: _summary([r for r in rows if r["category"] == c]) for c in categories},
        "per_aggregation": {
            a: _summary([r for r in rows if r["gold_aggregation"] == a])
            for a in AGGREGATIONS
            if any(r["gold_aggregation"] == a for r in rows)
        },
        "predictions": rows,
        "definitions": dict(METRIC_DEFINITIONS),
    }


def _baseline_result(columns: Sequence[str], rows: Sequence[Sequence[str]], coords, aggregation: str) -> dict[str, Any]:
    cells = [rows[r][c] for r, c in coords]
    value, unparsed = compute_numeric_answer(cells, aggregation)
    return {
        "cells": cells,
        "coordinates": [list(c) for c in coords],
        "aggregation": aggregation,
        "answer": (f"{aggregation} > " if aggregation != "NONE" else "") + ", ".join(cells),
        "numeric_answer": value,
        "unparsed_cells": unparsed,
        "decision_rule": DECISION_RULE,
    }


def first_cell_baseline(records: Sequence[Mapping[str, Any]]) -> dict[str, Any]:
    """Floor: answer every question with the first cell of the first column and NONE."""
    results = []
    for record in records:
        columns = list(record["table"])
        rows = [[record["table"][h][i] for h in columns] for i in range(len(record["table"][columns[0]]))]
        results.append(_baseline_result(columns, rows, [(0, 0)], "NONE"))
    return {**denotation_metrics(results, records), "baseline": "first cell of the first column, NONE"}


def keyword_lookup_baseline(records: Sequence[Mapping[str, Any]]) -> dict[str, Any]:
    """A non-neural heuristic: the answer column is the header sharing the most words with the question, the
    answer rows are those whose other cells appear verbatim in the question (else the first row), and the
    operator is read from the question's wording (``how many`` → COUNT, ``total``/``sum`` → SUM,
    ``average`` → AVERAGE, else NONE)."""
    results = []
    for record in records:
        columns = list(record["table"])
        rows = [[record["table"][h][i] for h in columns] for i in range(len(record["table"][columns[0]]))]
        question = record["question"].lower()
        q_words = _words(question)
        overlap = [len(_words(h) & q_words) for h in columns]
        sel = max(range(len(columns)), key=lambda i: (overlap[i], -i))
        hits = [
            r
            for r, row in enumerate(rows)
            if any(c != sel and len(str(cell).strip()) >= 2 and str(cell).strip().lower() in question for c, cell in enumerate(row))
        ]
        if not hits:
            hits = [0]
        if "how many" in question or question.startswith("count"):
            aggregation = "COUNT"
        elif "total" in question or "sum " in question:
            aggregation = "SUM"
        elif "average" in question or "mean " in question:
            aggregation = "AVERAGE"
        else:
            aggregation = "NONE"
        results.append(_baseline_result(columns, rows, [(r, sel) for r in hits], aggregation))
    return {**denotation_metrics(results, records), "baseline": "header-overlap column, verbatim-cell rows, wording-based operator"}

**Module 3/3:** `src/tapas_table_qa_pipeline/samples.py` (carried verbatim; see the note above)

In [ ]:
"""Labelled table-question datasets for the adaptation contract: the digest-pinned WikiSQL sample, the record
contract and its structural validation, table-disjoint splitting, and the BYOD JSONL loader.

A record is ``{id, table, question, answer}`` where ``table`` is ``{column: [cells]}`` (every header and cell a
str), ``question`` a non-empty str and ``answer`` the supervision in the pipeline's own vocabulary:
``{"aggregation": NONE|SUM|AVERAGE|COUNT, "coordinates": [[row, col], ...], "denotation": [cell, ...] | number}``.
For ``NONE`` the denotation is the selected cells; for the three operators it is the number the operator
should produce from the selected cells (the pipeline computes it the same way at inference). An optional
``category`` (free text, at most 32 characters) is carried into the per-category breakdown.

The default sample is drawn from the WikiSQL validation shard (BSD-3-Clause, Zhong et al. 2017) converted to
parquet by the Hugging Face Hub at an immutable revision: the shard is fetched whole (3.6 MB), refused on any
byte-count or SHA-256 mismatch, its SQL programme executed in pure Python to obtain the gold cells and value,
and a seeded, table-disjoint, aggregation-stratified draw taken. WikiSQL's MAX/MIN become ``NONE`` with the
extreme cell selected (TAPAS-WTQ has no MAX/MIN operator); its AVG is the pipeline's ``AVERAGE``. The original
operator is kept as ``category`` (``lookup``, ``max``, ``min``, ``count``, ``sum``, ``average``).
"""
# ruff: noqa: E501  -- record and pin literals are kept on single lines

from __future__ import annotations

import csv
import hashlib
import io
import json
import random
import re
import urllib.request
from collections.abc import Mapping, Sequence
from pathlib import Path
from typing import Any

# standalone rewrite (build_notebook.py): `from .pipeline import (` removed — names are kernel globals defined by the carried modules

CORPUS_NAME = "WikiSQL (validation shard)"
CORPUS_REPO = "Salesforce/wikisql"
CORPUS_REVISION = "48cfb60afd0d5f9d2231ca90f76edf9f975181bc"  # refs/convert/parquet commit on the Hub
CORPUS_FILE = "default/validation/0000.parquet"
CORPUS_SHA256 = "ed524cc7221e6c0010c7b57691da40897b38dce65090ecff48836989d606a347"
CORPUS_BYTES = 3_630_670
CORPUS_ROWS = 8_421
CORPUS_LICENSE = "BSD-3-Clause (Zhong, Xiong & Socher 2017, https://github.com/salesforce/WikiSQL)"
CORPUS_URL = f"https://huggingface.co/datasets/{CORPUS_REPO}/resolve/{CORPUS_REVISION}/{CORPUS_FILE}"
DEFAULT_CACHE_DIR = Path("weights") / "wikisql"

# Candidate filter (the sample must fit MAX_TOKENS untruncated so every gold cell is visible to the model).
SAMPLE_MAX_ROWS = 20
SAMPLE_MAX_COLUMNS = 12
SAMPLE_MAX_CHARS = 1_000  # header + cells
SAMPLE_MAX_LOOKUP_CELLS = 4
SAMPLE_SEED = 42
# Per WikiSQL operator. Training keeps lookups the majority (80 of 240) as they are in WikiSQL and in most table-QA
# use, so the adaptation does not trade lookup accuracy for operator accuracy; validation and test are stratified
# evenly so every operator is read on the same footing (90 / 150).
SAMPLE_SPLIT: dict[str, int | dict[str, int]] = {
    "train": {"lookup": 80, "count": 32, "sum": 32, "average": 32, "max": 32, "min": 32},
    "validation": 15,
    "test": 25,
}
SAMPLE_TABLE_FRACTIONS = {"train": 0.55, "validation": 0.2}  # of the candidate tables; the rest (0.25) is test
WIKISQL_AGGREGATIONS = ("NONE", "MAX", "MIN", "COUNT", "SUM", "AVG")
CATEGORY_OF = {"NONE": "lookup", "MAX": "max", "MIN": "min", "COUNT": "count", "SUM": "sum", "AVG": "average"}
SAMPLE_DIGEST = "806362262fe311efd659049821f857946ad7f4188385172f9c3d0411e919463e"  # dataset_digest over the three default splits together; tests pin it

MIN_RECORDS = 8
MAX_RECORDS = 20_000
MAX_CATEGORY_CHARS = 32
_ID_RE = re.compile(r"^[A-Za-z0-9_.:-]{1,64}$")


def _sha256_bytes(data: bytes) -> str:
    return hashlib.sha256(data).hexdigest()


def _normalise(text: str) -> str:
    return " ".join(str(text).lower().split())


# ---------------------------------------------------------------------------------------------------------
# WikiSQL: fetch, execute, draw
# ---------------------------------------------------------------------------------------------------------


def fetch_corpus(*, cache_dir: str | Path | None = None, fetcher: Any = None) -> bytes:
    """Return the pinned WikiSQL validation shard from the cache or the Hub; refuse any size/digest mismatch."""
    cache = Path(cache_dir) if cache_dir is not None else DEFAULT_CACHE_DIR
    cache.mkdir(parents=True, exist_ok=True)
    local = cache / "wikisql-validation.parquet"
    data = local.read_bytes() if local.is_file() else b""
    if len(data) != CORPUS_BYTES or _sha256_bytes(data) != CORPUS_SHA256:
        if fetcher is not None:
            data = fetcher(CORPUS_URL)
        else:
            request = urllib.request.Request(CORPUS_URL, headers={"User-Agent": "tapas-table-qa-pipeline"})
            with urllib.request.urlopen(request, timeout=120) as response:  # noqa: S310 - pinned https URL
                data = response.read()
        if len(data) != CORPUS_BYTES:
            raise ValueError(f"{CORPUS_FILE}: {len(data)} bytes, pinned {CORPUS_BYTES}")
        digest = _sha256_bytes(data)
        if digest != CORPUS_SHA256:
            raise ValueError(f"{CORPUS_FILE}: sha256 {digest} != pinned {CORPUS_SHA256}")
        local.write_bytes(data)
    return data


def read_corpus(data: bytes) -> list[dict[str, Any]]:
    """Decode the shard's ``question``, ``table`` and ``sql`` columns into plain dicts (index = shard row)."""
    import pyarrow.parquet as pq

    table = pq.read_table(io.BytesIO(data), columns=["question", "table", "sql"])
    if table.num_rows != CORPUS_ROWS:
        raise ValueError(f"{CORPUS_FILE}: {table.num_rows} rows, pinned {CORPUS_ROWS}")
    out = []
    for index, row in enumerate(table.to_pylist()):
        row["index"] = index
        out.append(row)
    return out


def _matching_rows(rows: Sequence[Sequence[str]], conds: Mapping[str, Sequence[Any]]) -> list[int]:
    keep = []
    for i, row in enumerate(rows):
        ok = True
        for col, op, cond in zip(conds["column_index"], conds["operator_index"], conds["condition"], strict=True):
            cell = row[col]
            if op == 0:
                ok = _normalise(cell) == _normalise(cond)
            else:
                a, b = parse_number(str(cell)), parse_number(str(cond))
                ok = a is not None and b is not None and (a > b if op == 1 else a < b)
            if not ok:
                break
        if ok:
            keep.append(i)
    return keep


def execute_sql(example: Mapping[str, Any]) -> dict[str, Any] | None:
    """Run one WikiSQL programme on its table. Returns the record's ``answer`` in the pipeline's vocabulary plus
    ``category``, or None when the programme has no usable denotation (no matching row, unparseable numbers)."""
    table, sql = example["table"], example["sql"]
    rows, sel, op = table["rows"], int(sql["sel"]), WIKISQL_AGGREGATIONS[int(sql["agg"])]
    matched = _matching_rows(rows, sql["conds"])
    if not matched:
        return None
    cells = [str(rows[r][sel]) for r in matched]
    coords = [[r, sel] for r in matched]
    category = CATEGORY_OF[op]
    if op == "NONE":
        return {"aggregation": "NONE", "coordinates": coords, "denotation": cells, "category": category}
    if op == "COUNT":
        return {"aggregation": "COUNT", "coordinates": coords, "denotation": float(len(cells)), "category": category}
    parsed = [parse_number(c) for c in cells]
    if any(v is None for v in parsed):
        return None
    if op in ("MAX", "MIN"):
        pick = max(range(len(parsed)), key=lambda i: parsed[i]) if op == "MAX" else min(range(len(parsed)), key=lambda i: parsed[i])
        return {"aggregation": "NONE", "coordinates": [coords[pick]], "denotation": [cells[pick]], "category": category}
    aggregation = "SUM" if op == "SUM" else "AVERAGE"
    value, _ = compute_numeric_answer(cells, aggregation)
    return {"aggregation": aggregation, "coordinates": coords, "denotation": float(value), "category": category}


def table_digest(table: Mapping[str, Sequence[str]]) -> str:
    """SHA-256 of the normalised header and cells: the identity a split is made disjoint on."""
    columns, rows = _check_table(table)
    payload = json.dumps([[_normalise(c) for c in columns], [[_normalise(v) for v in row] for row in rows]])
    return _sha256_bytes(payload.encode("utf-8"))


def wikisql_candidates(examples: Sequence[Mapping[str, Any]]) -> list[dict[str, Any]]:
    """Records for every shard example whose table fits the sample ceilings and whose programme executes."""
    out = []
    for ex in examples:
        tb = ex["table"]
        header, rows = [str(h) for h in tb["header"]], [[str(c) for c in row] for row in tb["rows"]]
        if not 1 <= len(rows) <= SAMPLE_MAX_ROWS or not 1 <= len(header) <= SAMPLE_MAX_COLUMNS:
            continue
        if any(len(c) > 200 for row in rows for c in row) or any(len(h) > 200 for h in header):
            continue
        if sum(len(c) for row in rows for c in row) + sum(len(h) for h in header) > SAMPLE_MAX_CHARS:
            continue
        if len(set(header)) != len(header):
            continue
        answer = execute_sql({"table": {"rows": rows}, "sql": ex["sql"]})
        if answer is None or (answer["aggregation"] == "NONE" and len(answer["coordinates"]) > SAMPLE_MAX_LOOKUP_CELLS):
            continue
        category = answer.pop("category")
        out.append(
            {
                "id": f"wikisql-val-{ex['index']}",
                "table": {h: [row[i] for row in rows] for i, h in enumerate(header)},
                "question": str(ex["question"]),
                "answer": answer,
                "category": category,
                "source_table_id": str(tb["id"]),
                "sql": str(ex["sql"]["human_readable"]),
            }
        )
    return out


def build_sample_dataset(
    candidates: Sequence[Mapping[str, Any]],
    *,
    seed: int = SAMPLE_SEED,
    sizes: Mapping[str, int | Mapping[str, int]] | None = None,
) -> dict[str, list[dict[str, Any]]]:
    """Seeded draw: tables are shuffled and assigned to a split first (so no table crosses splits), then
    ``sizes[split]`` records per WikiSQL operator (one int for all, or a per-category mapping) are drawn from
    each split's pool."""
    sizes = dict(sizes or SAMPLE_SPLIT)
    rng = random.Random(seed)
    tables = sorted({str(c["source_table_id"]) for c in candidates})
    rng.shuffle(tables)
    n = len(tables)
    train_end = int(n * SAMPLE_TABLE_FRACTIONS["train"])
    val_end = train_end + int(n * SAMPLE_TABLE_FRACTIONS["validation"])
    split_of = {t: ("train" if i < train_end else "validation" if i < val_end else "test") for i, t in enumerate(tables)}
    pools: dict[str, dict[str, list[dict[str, Any]]]] = {name: {} for name in sizes}
    for c in candidates:
        pools[split_of[str(c["source_table_id"])]].setdefault(str(c["category"]), []).append(dict(c))
    out: dict[str, list[dict[str, Any]]] = {}
    for name, spec in sizes.items():
        drawn = []
        for category in sorted(pools[name]):
            per = int(spec[category]) if isinstance(spec, Mapping) else int(spec)
            items = pools[name][category]
            rng.shuffle(items)
            if len(items) < per:
                raise ValueError(f"{name}/{category}: only {len(items)} candidates, need {per}")
            drawn.extend(items[:per])
        rng.shuffle(drawn)
        out[name] = drawn
    return out


def fetch_sample_dataset(*, cache_dir: str | Path | None = None, seed: int = SAMPLE_SEED) -> dict[str, list[dict[str, Any]]]:
    """fetch → read → candidates → draw, in one call."""
    return build_sample_dataset(wikisql_candidates(read_corpus(fetch_corpus(cache_dir=cache_dir))), seed=seed)


# ---------------------------------------------------------------------------------------------------------
# Record contract
# ---------------------------------------------------------------------------------------------------------


def _check_answer(answer: Any, columns: Sequence[str], rows: Sequence[Sequence[str]], where: str) -> dict[str, Any]:
    if not isinstance(answer, Mapping):
        raise ValueError(f"{where}: answer must be a mapping with aggregation/coordinates/denotation")
    aggregation = answer.get("aggregation")
    if aggregation not in AGGREGATIONS:
        raise ValueError(f"{where}: answer.aggregation must be one of {AGGREGATIONS}, got {aggregation!r}")
    coords = answer.get("coordinates")
    if not isinstance(coords, Sequence) or isinstance(coords, str | bytes) or not coords:
        raise ValueError(f"{where}: answer.coordinates must be a non-empty list of [row, col] pairs")
    checked_coords = []
    for pair in coords:
        if not isinstance(pair, Sequence) or isinstance(pair, str | bytes) or len(pair) != 2:
            raise ValueError(f"{where}: answer.coordinates entries must be [row, col] pairs")
        r, c = pair
        if isinstance(r, bool) or isinstance(c, bool) or not isinstance(r, int) or not isinstance(c, int):
            raise ValueError(f"{where}: answer.coordinates must be integer pairs")
        if not (0 <= r < len(rows) and 0 <= c < len(columns)):
            raise ValueError(f"{where}: coordinate {[r, c]} outside the {len(rows)}x{len(columns)} table")
        if [r, c] in checked_coords:
            raise ValueError(f"{where}: duplicate coordinate {[r, c]}")
        checked_coords.append([r, c])
    denotation = answer.get("denotation")
    cells = [rows[r][c] for r, c in checked_coords]
    if aggregation == "NONE":
        if not isinstance(denotation, Sequence) or isinstance(denotation, str | bytes) or not all(isinstance(d, str) for d in denotation):
            raise ValueError(f"{where}: a NONE answer's denotation must be a list of cell strings")
        if sorted(_normalise(d) for d in denotation) != sorted(_normalise(c) for c in cells):
            raise ValueError(f"{where}: NONE denotation {list(denotation)} does not match the selected cells {cells}")
        checked_denotation: Any = [str(d) for d in denotation]
    else:
        if isinstance(denotation, bool) or not isinstance(denotation, int | float):
            raise ValueError(f"{where}: a {aggregation} answer's denotation must be a number")
        value, unparsed = compute_numeric_answer(cells, aggregation)
        if value is None:
            raise ValueError(f"{where}: {aggregation} over cells that do not parse as numbers: {unparsed}")
        if abs(float(value) - float(denotation)) > 1e-6:
            raise ValueError(f"{where}: {aggregation} of the selected cells is {value}, denotation says {denotation}")
        checked_denotation = float(denotation)
    return {"aggregation": aggregation, "coordinates": checked_coords, "denotation": checked_denotation}


def _check_record(record: Any, index: int) -> dict[str, Any]:
    where = f"records[{index}]"
    if not isinstance(record, Mapping):
        raise ValueError(f"{where} must be a mapping with id/table/question/answer")
    for key in ("id", "table", "question", "answer"):
        if key not in record:
            raise ValueError(f"{where} is missing {key!r}")
    rid = record["id"]
    if not isinstance(rid, str) or not _ID_RE.match(rid):
        raise ValueError(f"{where}: id must match {_ID_RE.pattern}")
    columns, rows = _check_table(record["table"])
    question = _check_query(record["question"])
    answer = _check_answer(record["answer"], columns, rows, where)
    item = {"id": rid, "table": {h: [row[i] for row in rows] for i, h in enumerate(columns)}, "question": question, "answer": answer}
    category = record.get("category")
    if category is not None:
        if not isinstance(category, str) or not 1 <= len(category.strip()) <= MAX_CATEGORY_CHARS:
            raise ValueError(f"{where}: category must be a str of 1..{MAX_CATEGORY_CHARS} characters")
        item["category"] = category.strip()
    for key in ("source_table_id", "sql"):
        if key in record:
            item[key] = str(record[key])
    return item


def validate_dataset(
    records: Sequence[Mapping[str, Any]],
    *,
    min_records: int = MIN_RECORDS,
    max_records: int = MAX_RECORDS,
) -> dict[str, Any]:
    """Structural validation of a labelled table-question dataset; raises ValueError before any model import."""
    if isinstance(records, Mapping) or not isinstance(records, Sequence) or isinstance(records, str | bytes):
        raise ValueError("records must be a list of {id, table, question, answer} mappings")
    if not min_records <= len(records) <= max_records:
        raise ValueError(f"{len(records)} records; {min_records}..{max_records} are required")
    checked, ids, aggregations, categories, tables = [], set(), {}, {}, set()
    for index, record in enumerate(records):
        item = _check_record(record, index)
        if item["id"] in ids:
            raise ValueError(f"duplicate id {item['id']!r}")
        ids.add(item["id"])
        aggregations[item["answer"]["aggregation"]] = aggregations.get(item["answer"]["aggregation"], 0) + 1
        if "category" in item:
            categories[item["category"]] = categories.get(item["category"], 0) + 1
        tables.add(table_digest(item["table"]))
        checked.append(item)
    return {
        "records": checked,
        "n_records": len(checked),
        "n_tables": len(tables),
        "aggregation_counts": dict(sorted(aggregations.items())),
        "category_counts": dict(sorted(categories.items())),
        "table_shape": {
            "rows": [min(len(next(iter(r["table"].values()))) for r in checked), max(len(next(iter(r["table"].values()))) for r in checked)],
            "columns": [min(len(r["table"]) for r in checked), max(len(r["table"]) for r in checked)],
        },
        "digest": dataset_digest(checked),
        "model_id": MODEL_ID,
    }


def dataset_digest(records: Sequence[Mapping[str, Any]]) -> str:
    """Order-independent SHA-256 over (id, table digest, question, answer)."""
    parts = sorted(
        json.dumps([r["id"], table_digest(r["table"]), r["question"], r["answer"]], sort_keys=True) for r in records
    )
    return _sha256_bytes("\n".join(parts).encode("utf-8"))


def check_split_disjoint(splits: Mapping[str, Sequence[Mapping[str, Any]]]) -> dict[str, Any]:
    """Assert no table (by normalised content) appears in two splits (leakage check)."""
    seen: dict[str, str] = {}
    for name, records in splits.items():
        for record in records:
            key = table_digest(record["table"])
            if key in seen and seen[key] != name:
                raise ValueError(f"table of {record['id']!r} appears in both {seen[key]} and {name}")
            seen[key] = name
    return {name: len(records) for name, records in splits.items()}


def split_dataset(
    records: Sequence[Mapping[str, Any]],
    *,
    val_fraction: float = 0.15,
    test_fraction: float = 0.2,
    seed: int = 0,
) -> dict[str, list[dict[str, Any]]]:
    """Seeded table-disjoint split of a BYOD dataset: tables are shuffled and cut by fraction, and every record of
    a table follows its table."""
    if not (0.0 <= val_fraction < 1.0 and 0.0 < test_fraction < 1.0 and val_fraction + test_fraction < 1.0):
        raise ValueError("fractions must satisfy 0 <= val < 1, 0 < test < 1, val + test < 1")
    checked = validate_dataset(records)["records"]
    by_table: dict[str, list[dict[str, Any]]] = {}
    for record in checked:
        by_table.setdefault(table_digest(record["table"]), []).append(record)
    keys = sorted(by_table)
    random.Random(seed).shuffle(keys)
    n = len(keys)
    n_test = max(1, round(n * test_fraction))
    n_val = round(n * val_fraction)
    if n - n_test - n_val < 1:
        raise ValueError(f"{n} distinct tables are too few to split into train/validation/test")
    out = {"train": [], "validation": [], "test": []}
    for i, key in enumerate(keys):
        name = "test" if i < n_test else "validation" if i < n_test + n_val else "train"
        out[name].extend(by_table[key])
    return out


def load_byod_dataset(path: str | Path) -> list[dict[str, Any]]:
    """Read records from a JSONL file (one record per line) or a JSON list; validation happens downstream."""
    text = Path(path).read_text(encoding="utf-8")
    stripped = text.strip()
    if stripped.startswith("["):
        try:
            data = json.loads(stripped)  # one JSON list of records
        except json.JSONDecodeError:
            data = None  # not a single document: read it as JSONL below
        if isinstance(data, list):
            if any(not isinstance(r, Mapping) for r in data):
                raise ValueError("each JSONL line must be a record object")
            return data
    rows = [json.loads(line) for line in text.splitlines() if line.strip()]
    if any(not isinstance(r, Mapping) for r in rows):
        raise ValueError("each JSONL line must be a record object")
    return rows


def write_dataset_jsonl(records: Sequence[Mapping[str, Any]], path: str | Path) -> Path:
    """Write records as JSONL in the BYOD shape."""
    out = Path(path)
    out.parent.mkdir(parents=True, exist_ok=True)
    with open(out, "w", encoding="utf-8") as handle:
        for record in records:
            row = {k: record[k] for k in ("id", "table", "question", "answer") if k in record}
            if "category" in record:
                row["category"] = record["category"]
            handle.write(json.dumps(row, ensure_ascii=False) + "\n")
    return out


def write_dataset_csv(records: Sequence[Mapping[str, Any]], path: str | Path) -> Path:
    """A human-readable summary (id, category, aggregation, question, sql, denotation), not the BYOD format."""
    out = Path(path)
    out.parent.mkdir(parents=True, exist_ok=True)
    with open(out, "w", encoding="utf-8", newline="") as handle:
        writer = csv.writer(handle)
        writer.writerow(["id", "category", "aggregation", "question", "sql", "denotation"])
        for r in records:
            d = r["answer"]["denotation"]
            writer.writerow([r["id"], r.get("category", ""), r["answer"]["aggregation"], r["question"], r.get("sql", ""), json.dumps(d) if isinstance(d, list) else d])
    return out

## 3. Pin, stage and verify the model

The model identity is carried twice — `MODEL_ID`/`MODEL_REVISION` in the module above and the `6`-file manifest below (paths, byte sizes, SHA-256) — and the cell first asserts they agree. It writes the manifest into the working-directory snapshot, then `stage_missing_files(..., allow_download=True)` fetches exactly the entries that are absent from the Hugging Face Hub **at revision `f58317ab2577…`** (never `main`), `verify_snapshot` re-hashes every file and raises on the first size or digest mismatch, and only then does `TAPASTableQAPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)` load the verified files. There is no fallback to a different download and no remote model code is executed. The effective identity, device and weight source are printed before any inference.

In [ ]:
import json

MANIFEST = {
  "format": "dimer_hf_snapshot",
  "formatVersion": 1,
  "modelKey": "tapas-large-wtq",
  "modelId": "google/tapas-large-finetuned-wtq",
  "revision": "f58317ab2577d17647d9acafa790c744a0388b30",
  "files": [
    {
      "path": "README.md",
      "bytes": 7224,
      "sha256": "2397bdae46316684903465e6f425a9fe10c85d491a9755ec7a31a74353c034c8"
    },
    {
      "path": "config.json",
      "bytes": 1659,
      "sha256": "834f44a325a349d2d209ecb18e4fa2ca3cdb16cef5b169e5c0a8f5e16c444d13"
    },
    {
      "path": "model.safetensors",
      "bytes": 1346985282,
      "sha256": "149247e13732c222ba621e0c4e7b90ba260869b36adcbe115dc872c2c98bccf0"
    },
    {
      "path": "special_tokens_map.json",
      "bytes": 154,
      "sha256": "e3ec7abc6bcd45aba696cb95e9945186dad920aea78733f8b45c83d696ed0dea"
    },
    {
      "path": "tokenizer_config.json",
      "bytes": 490,
      "sha256": "ab033df4f3c902cbc66bae2736bc99f4a59c43e7d9e53e40046576826b2cc5e9"
    },
    {
      "path": "vocab.txt",
      "bytes": 262028,
      "sha256": "4d96f9308bcf9019684fcc109aa8c042b9b745edabba0162fbe66c75ebee2db4"
    }
  ],
  "totalBytes": 1347256837
}

if (MANIFEST['modelId'], MANIFEST['revision']) != (MODEL_ID, MODEL_REVISION):
    raise RuntimeError('inline manifest does not name the identity carried by the pipeline module; the notebook was not regenerated after a change')
WEIGHTS_DIR = DEFAULT_WEIGHTS_DIR
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
with open(WEIGHTS_DIR / MANIFEST_NAME, 'w', encoding='utf-8') as handle:
    json.dump(MANIFEST, handle, indent=2)
print({'model_id': MODEL_ID, 'revision': MODEL_REVISION, 'license': MODEL_LICENSE, 'files': len(MANIFEST['files']), 'total_bytes': MANIFEST['totalBytes']})
fetched = stage_missing_files(WEIGHTS_DIR, allow_download=True)
print({'weights_dir': str(WEIGHTS_DIR), 'fetched': fetched})
snapshot = verify_snapshot(WEIGHTS_DIR)
_files = snapshot.get('files', []) if isinstance(snapshot, dict) else []
print({'verified_files': len(_files) if isinstance(_files, list) else _files, 'revision': snapshot.get('revision', MODEL_REVISION) if isinstance(snapshot, dict) else MODEL_REVISION})
pipe = TAPASTableQAPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)
print({'device': getattr(pipe, 'device', None), 'source': getattr(pipe, 'source', 'local-snapshot')})

## 4. WikiSQL sample and split

`fetch_corpus` returns the pinned shard from the cache under `weights/wikisql/` or the Hub — every cached or fetched copy is refused on a byte-count or SHA-256 mismatch — and `read_corpus` decodes its `question`, `table` and `sql` columns (8,421 questions over 2,630 tables). `wikisql_candidates` keeps the questions whose table fits the sample ceilings (at most 20 rows, 12 columns and 1,000 characters, so nothing is truncated at 512 tokens) and whose SQL programme executes to a usable answer, mapping `MAX`/`MIN` to a `NONE` lookup of the extreme cell and `AVG` to `AVERAGE`; `build_sample_dataset` shuffles the tables, assigns each to one split first, then draws 80 lookups and 32 of each operator for training and 15 / 25 per operator for validation / test (240 / 90 / 150). `validate_dataset` checks every record against the contract — including that each operator denotation is what the pipeline's arithmetic produces from the gold cells — `check_split_disjoint` asserts no table (by normalised content) is shared, and the training split is written to `outputs/tapas_table_qa_train.jsonl` in the BYOD shape.

Look for: 4,972 candidates, the six categories with 80 / 32 in training and 15 / 25 elsewhere, three digests, and four refusal probes — a duplicate id, a coordinate outside the table, an operator denotation that disagrees with its cells, and a dataset too small to use — each rejected before the model does anything.

In [ ]:
import csv
import hashlib
import json
import time

USE_BYOD = False  # @param {type:"boolean"}
SPLIT_SEED = 42  # @param {type:"integer"}

os.makedirs('outputs', exist_ok=True)
if USE_BYOD:
    from google.colab import files
    uploaded = files.upload()
    file_name, payload = next(iter(uploaded.items()))
    byod_path = Path('work') / 'byod.jsonl'
    byod_path.parent.mkdir(parents=True, exist_ok=True)
    byod_path.write_bytes(payload)
    records = load_byod_dataset(byod_path)
    splits = split_dataset(records, seed=SPLIT_SEED)
    data_source = 'BYOD (' + file_name + ')'
    raw_rows = {'byod': len(records)}
else:
    shard = fetch_corpus(cache_dir='weights/wikisql')
    corpus = read_corpus(shard)
    candidates = wikisql_candidates(corpus)
    splits = build_sample_dataset(candidates, seed=SPLIT_SEED)
    data_source = f'{CORPUS_NAME}: {CORPUS_REPO} @ {CORPUS_REVISION[:12]} ({CORPUS_LICENSE})'
    raw_rows = {'shard_bytes': len(shard), 'questions': len(corpus), 'tables': len({r['table']['id'] for r in corpus}), 'candidates': len(candidates)}
dataset_manifests = {name: validate_dataset(part) for name, part in splits.items()}
splits = {name: manifest['records'] for name, manifest in dataset_manifests.items()}
disjoint = check_split_disjoint(splits)
train_records, val_records, test_records = splits['train'], splits['validation'], splits['test']
write_dataset_jsonl(train_records, 'outputs/tapas_table_qa_train.jsonl')
print({'data_source': data_source, 'raw_rows': raw_rows, 'splits': disjoint})
for name, manifest in dataset_manifests.items():
    print({name: {'n': manifest['n_records'], 'tables': manifest['n_tables'], 'categories': manifest['category_counts'], 'aggregations': manifest['aggregation_counts'], 'table_shape': manifest['table_shape'], 'digest': manifest['digest'][:16] + '...'}})
example = train_records[0]
print({'example': {'id': example['id'], 'category': example.get('category'), 'question': example['question'], 'sql': example.get('sql'), 'answer': example['answer'], 'columns': list(example['table']), 'rows': len(next(iter(example['table'].values())))}})

probes = {
    'duplicate id': [{**r, 'id': 'same'} for r in train_records[:8]],
    'coordinate outside the table': [{**train_records[0], 'answer': {**train_records[0]['answer'], 'coordinates': [[999, 0]]}}, *train_records[1:8]],
    'operator denotation disagrees with its cells': [{**r, 'answer': {**r['answer'], 'denotation': 1e9}} if r['answer']['aggregation'] != 'NONE' else r for r in train_records[:8]],
    'too small': train_records[:3],
}
for name, probe in probes.items():
    try:
        validate_dataset(probe)
        print({'probe': name, 'verdict': 'accepted'})
    except (TypeError, ValueError) as exc:
        print({'probe': name, 'rejected': str(exc)[:110]})

## 5. Answer the synthetic table through the inference contract

The inference contract is exercised as the inference-only tutorial exercised it: a 4-row × 3-column table of Philippine cities authored in code (the model card's smoke table, population cells with thousands separators) and three questions — a **lookup**, a **COUNT** and a **SUM** — with the answers the author expects; a different table family from WikiSQL, and questions the adapted model will answer again in Section 9. `validate_inputs` applies exactly the checks `answer` applies (`MAX_ROWS`, `MAX_COLUMNS`, `MAX_CELL_CHARS`, `MAX_QUERY_CHARS`; `MAX_TOKENS` is applied by the tokenizer and *truncates*, reported by `answer`) and returns an input manifest; a table carrying a numeric (non-string) cell is validated too and its rejection recorded as a finding. `answer(table, query)` returns `cells`, `coordinates`, `aggregation`, `numeric_answer` with `numeric_answer_source`, `unparsed_cells`, `aggregation_logits`, `n_tokens`, `tokens_before_truncation`, `rows_kept`, `truncated` and the decision rule; the per-grid `evaluation_report` on three authored questions is `sample-sanity` — plumbing evidence, not a measurement; whether the model is *right* is what Section 6 measures on 150 questions.

In [ ]:
ceilings = {'MAX_ROWS': MAX_ROWS, 'MAX_COLUMNS': MAX_COLUMNS, 'MAX_TOKENS': MAX_TOKENS, 'MAX_QUERY_CHARS': MAX_QUERY_CHARS, 'MAX_CELL_CHARS': MAX_CELL_CHARS, 'AGGREGATIONS': AGGREGATIONS, 'CELL_THRESHOLD': CELL_THRESHOLD, 'MIN_RECORDS': MIN_RECORDS, 'MAX_RECORDS': MAX_RECORDS, 'device': pipe.device}
print(ceilings)
table = {
    'City': ['Manila', 'Cebu', 'Davao', 'Baguio'],
    'Population (2020)': ['1,846,513', '964,169', '1,776,949', '366,358'],
    'Region': ['NCR', 'Region VII', 'Region XI', 'CAR'],
}
queries = ['Which city is in Region VII?', 'How many cities are listed?', 'What is the total population of Manila and Davao?']
golds = ['Cebu', 4, 3623462]
query_ids = [f'q{index + 1}' for index in range(len(queries))]
sample_sha256 = hashlib.sha256(json.dumps({'table': table, 'queries': queries}, sort_keys=True).encode('utf-8')).hexdigest()
input_manifest = validate_inputs(table, queries, names=query_ids)
try:
    validate_inputs({'Item': ['a', 'b'], 'Units': [1, 2]}, queries)
except TypeError as exc:
    input_manifest['findings'].append({'input': 'numeric-cell-probe', 'verdict': 'rejected', 'message': str(exc)})
with open('outputs/tapas_table_qa_input_manifest.json', 'w', encoding='utf-8') as handle:
    json.dump(input_manifest, handle, indent=2, ensure_ascii=False)
print({'sample_sha256': sample_sha256[:16] + '...', 'manifest_verdict': input_manifest['verdict'], 'findings': len(input_manifest['findings'])})


def answer_table(pipeline, label):
    rows = []
    n_rows = len(next(iter(table.values())))
    for query_id, query in zip(query_ids, queries, strict=True):
        started = time.perf_counter()
        result = pipeline.answer(table, query)
        elapsed = time.perf_counter() - started
        checks = {
            'aggregation_known': result['aggregation'] in AGGREGATIONS,
            'coordinates_inside_table': all(0 <= r < n_rows and 0 <= c < len(table) for r, c in result['coordinates']),
            'cells_match_coordinates': result['cells'] == [list(table.values())[c][r] for r, c in result['coordinates']],
            'count_equals_cells': result['aggregation'] != 'COUNT' or result['numeric_answer'] == len(result['cells']),
            'n_tokens_within_ceiling': 1 <= result['n_tokens'] <= MAX_TOKENS,
        }
        if not all(checks.values()):
            raise RuntimeError(f'answer output failed a sanity check for {query_id}: {checks}')
        rows.append({'id': query_id, 'query': query, 'model': label, 'seconds': round(elapsed, 3), 'checks': checks, **result})
        print(f"{label} {query_id}: {result['aggregation']:<8} cells={result['cells']} numeric_answer={result['numeric_answer']} tokens={result['n_tokens']} truncated={result['truncated']} seconds={elapsed:.2f}")
    return rows


frozen_rows = answer_table(pipe, 'frozen')
frozen_scene = evaluation_report(frozen_rows, golds, sample_kind='synthetic (authored in this notebook; the model card smoke table)')
print({'decision_rule': DECISION_RULE, 'numeric_answer_source': NUMERIC_ANSWER_SOURCE})
print({'frozen_scene': {m['id']: round(m['value'], 3) for m in frozen_scene['metrics']}, 'verdict': frozen_scene['verdict']})

## 6. Baselines and the frozen model on the test questions

Three systems frame the adaptation, each read three ways by `denotation_metrics` (carried in `metrics.py`): **denotation accuracy** (the WTQ criterion — a number to 1e-6 for an operator, otherwise the same multiset of cell strings), **aggregation accuracy** (the predicted operator equals the record's) and **cell accuracy** (the selected coordinates equal the record's as a set), overall and per question type. The **first-cell floor** answers every question with the first cell of the first column and `NONE`. The **keyword lookup** picks the column whose header shares the most words with the question, the rows whose other cells appear verbatim in it, and an operator from the question's wording (`how many` → `COUNT`, `total` → `SUM`, `average` → `AVERAGE`) — a table-QA system that never sees a token embedding. The **frozen model** is scored by `pipe.evaluate`, which answers every record through `answer` and scores the results. Expect the frozen model far above both baselines on denotation and lookups near the ceiling, and read the operator column: the build record measured 0.82 / 0.57 / 0.82 frozen, with `count` at 0.64 and lookups at 0.92.

In [ ]:
METRICS = ('accuracy', 'aggregation_accuracy', 'cell_accuracy')
baseline_first = first_cell_baseline(test_records)
baseline_keyword = keyword_lookup_baseline(test_records)
print({'first_cell_baseline': {k: round(baseline_first[k], 3) for k in METRICS}, 'n': baseline_first['n'], 'note': baseline_first['baseline']})
print({'keyword_lookup_baseline': {k: round(baseline_keyword[k], 3) for k in METRICS}, 'by_category': {c: round(v['accuracy'], 2) for c, v in baseline_keyword['per_category'].items()}, 'note': baseline_keyword['baseline']})
t0 = time.perf_counter()
frozen_test = pipe.evaluate(test_records)
print({'frozen_model_test': {k: round(frozen_test[k], 3) for k in METRICS}, 'n': frozen_test['n'], 'verdict': frozen_test['verdict'], 'truncated': frozen_test['truncated'], 'seconds': round(time.perf_counter() - t0, 1)})
print({'definitions': frozen_test['definitions']})
frozen_fields = {c: {'n': v['n'], 'accuracy': round(v['accuracy'], 2), 'aggregation': round(v['aggregation_accuracy'], 2), 'cells': round(v['cell_accuracy'], 2)} for c, v in frozen_test['per_category'].items()}
print({'by_category_frozen': frozen_fields})
assert frozen_test['accuracy'] > baseline_keyword['accuracy'] > baseline_first['accuracy']

## 7. Bounded fine-tuning of the last encoder blocks and the heads

`pipe.adapt` trains only the last `TRAINABLE_LAYERS` encoder blocks and the three heads — the cell-selection weights, the column-selection weights and the aggregation classifier; two blocks by default, 25,198,598 of 336,734,214 parameters — while the embeddings and every earlier block stay frozen. Supervision is the checkpoint's own **weak supervision**: a lookup labels its gold cells (the tokenizer turns the coordinates into token labels); an operator question carries only its number as `float_answer`, so the model's regression loss trains the operator and the soft cell selection together — labelling an operator question's cells too would let the model's aggregate mask re-route it to cell selection whenever it prefers `NONE`, which is the failure being adapted away. AdamW at a fixed learning rate, gradient clipping at 1.0, seeded shuffling, batches padded to their longest table, no scheduler. Epoch 0 records the frozen model's validation metrics; every epoch is scored on the 90 validation questions and the epoch with the highest validation **score** — the mean of denotation, aggregation and cell accuracy, a steadier selector than denotation accuracy alone on 90 questions — is kept.

Watch the training loss fall from about 3 to below 0.5 within four epochs while the validation aggregation accuracy jumps in the first epoch: the operator choice is what these questions teach. The build record's counter-examples — four blocks at the same rate, or a training draw with as many operator questions as lookups — traded lookup accuracy for operator accuracy; the default is the configuration that kept lookups whole.

In [ ]:
EPOCHS = 4  # @param {type:"integer"}
LEARNING_RATE = 5e-5  # @param {type:"number"}
BATCH_SIZE = 8  # @param {type:"integer"}
TRAINABLE_LAYERS = 2  # @param {type:"integer"}


def report(entry):
    row = {'epoch': entry['epoch'], 'train_loss': None if entry['train_loss'] is None else round(entry['train_loss'], 4)}
    if entry.get('val'):
        row.update({'val_' + k: round(entry['val'][k], 3) for k in (*METRICS, 'score')})
    if 'note' in entry:
        row['note'] = entry['note']
    print(row)


t0 = time.perf_counter()
adapt_result = pipe.adapt(train_records, val_records, epochs=EPOCHS, lr=LEARNING_RATE, batch_size=BATCH_SIZE, trainable_layers=TRAINABLE_LAYERS, progress=report)
adapt_seconds = round(time.perf_counter() - t0, 1)
print({'trainable_parameters': adapt_result['n_trainable'], 'total_parameters': adapt_result['n_total'], 'best_epoch': adapt_result['best_epoch'], 'selection': adapt_result['selection'], 'seconds': adapt_seconds})

## 8. Held-out evaluation

The test questions' tables were never used for training or epoch selection, and no table appears in two splits. The adapted model is scored exactly as the frozen model was in Section 6, the four systems are put side by side on the three measures, and the per-type breakdown is repeated. Read it in this order: **aggregation accuracy** first (what the adaptation teaches — the build record measured 0.57 → 0.81), then **cell accuracy** (0.82 → 0.85) and **denotation accuracy** (0.82 → 0.84, three questions of 150), then the per-type rows, where `average`, `min` and `sum` each gained one question, `count` and `max` did not move and lookups stayed at 0.92. The cell asserts the adapted aggregation accuracy is above the frozen one; denotation accuracy is reported, not asserted, because on 150 questions it moves by single questions and the build record's other draws moved it either way — the CPU pre-flight of this very notebook kept it at 0.82 (epoch 3 selected, aggregation accuracy 0.57 → 0.83) where the GPU run gained three questions (epoch 2, 0.57 → 0.81). One seeded split gives **no dispersion estimate**; the deltas are sample-sanity evidence that the adaptation contract works, not a benchmark, and a gain on six question types over Wikipedia tables says nothing about your tables until you measure them.

In [ ]:
adapted_test = pipe.evaluate(test_records)
adapted_val = pipe.evaluate(val_records)
adapted_fields = {c: {'n': v['n'], 'accuracy': round(v['accuracy'], 2), 'aggregation': round(v['aggregation_accuracy'], 2), 'cells': round(v['cell_accuracy'], 2)} for c, v in adapted_test['per_category'].items()}
comparison = {metric: {'first_cell': round(baseline_first[metric], 3), 'keyword': round(baseline_keyword[metric], 3), 'frozen': round(frozen_test[metric], 3), 'adapted': round(adapted_test[metric], 3)} for metric in METRICS}
comparison['delta_vs_frozen'] = {metric: round(adapted_test[metric] - frozen_test[metric], 3) for metric in METRICS}
comparison['by_category'] = {c: {'n': frozen_fields[c]['n'], 'frozen': frozen_fields[c]['accuracy'], 'adapted': adapted_fields[c]['accuracy'], 'frozen_aggregation': frozen_fields[c]['aggregation'], 'adapted_aggregation': adapted_fields[c]['aggregation']} for c in frozen_fields}
for key, row in comparison.items():
    print({key: row})
evaluation_report_payload = {
    'model': {'id': MODEL_ID, 'revision': MODEL_REVISION, 'key': MODEL_KEY},
    'data_source': data_source,
    'dataset_digests': {name: manifest['digest'] for name, manifest in dataset_manifests.items()},
    'splits': disjoint,
    'baselines': {'first_cell': {k: v for k, v in baseline_first.items() if k != 'predictions'}, 'keyword_lookup': {k: v for k, v in baseline_keyword.items() if k != 'predictions'}},
    'frozen_test': frozen_test,
    'validation_metrics': adapted_val,
    'test_metrics': adapted_test,
    'comparison': comparison,
    'adaptation': {k: v for k, v in adapt_result.items() if k not in ('history', 'trainable_names')},
    'history': adapt_result['history'],
    'adaptation_seconds': adapt_seconds,
}
with open('outputs/tapas_table_qa_evaluation_report.json', 'w', encoding='utf-8') as f:
    json.dump(evaluation_report_payload, f, indent=2, ensure_ascii=False)
assert adapted_test['aggregation_accuracy'] > frozen_test['aggregation_accuracy']
print({'report': 'outputs/tapas_table_qa_evaluation_report.json'})

## 9. Re-answer the synthetic table, export the adapter and reload it

The three city-table questions from Section 5 are answered again by the adapted model — a table family the adaptation never saw, so this is a small look at what the adaptation did *outside* its corpus (the build record's answers are in the model card; a changed answer here is a finding to record, not a failure) — reported with the per-grid `evaluation_report` (`sample-sanity`), and both answer sets are written to `outputs/tapas_table_qa_answers.csv`.

`pipe.save_artifact` writes the trained tensors — the last two encoder blocks and the three heads, about 100 MB — as `adapter.safetensors`, with a `manifest.json` recording the artifact format, the base model id and revision, the digest of the base `model.safetensors`, the tensor names, the file size and SHA-256, the training configuration and the epoch history (OUT8). `TAPASTableQAPipeline.from_artifact` re-verifies the base snapshot, checks the artifact manifest, its digest and its exact tensor set **before** deserialising, refuses any tensor outside the encoder blocks and heads, and overlays the tensors onto a freshly loaded base — a new object from files, not the in-memory model (VER2). The cell asserts identical cells, operators and aggregation logits on eight test questions (VER4).

In [ ]:
import shutil

adapted_rows = answer_table(pipe, 'adapted')
adapted_scene = evaluation_report(adapted_rows, golds, sample_kind='synthetic (authored in this notebook; the model card smoke table)')
print({'scene_after_adaptation': {m['id']: round(m['value'], 3) for m in adapted_scene['metrics']}, 'verdict': adapted_scene['verdict']})
with open('outputs/tapas_table_qa_answers.csv', 'w', encoding='utf-8', newline='') as handle:
    writer = csv.writer(handle)
    writer.writerow(['model', 'id', 'query', 'aggregation', 'cells', 'coordinates', 'numeric_answer', 'numeric_answer_source', 'unparsed_cells', 'n_tokens', 'rows_kept', 'truncated', 'seconds'])
    for r in [*frozen_rows, *adapted_rows]:
        writer.writerow([r['model'], r['id'], r['query'], r['aggregation'], ' | '.join(r['cells']), json.dumps(r['coordinates']), r['numeric_answer'], r['numeric_answer_source'], ' | '.join(r['unparsed_cells']), r['n_tokens'], r['rows_kept'], r['truncated'], r['seconds']])

artifact_dir = Path('outputs/tapas_table_qa_adapter')
shutil.rmtree(artifact_dir, ignore_errors=True)
pipe.save_artifact(artifact_dir, metadata={'tutorial': 'tapas_table_qa', 'data_source': data_source})
artifact_manifest = json.loads((artifact_dir / 'manifest.json').read_text(encoding='utf-8'))
print({'artifact': str(artifact_dir), 'format': artifact_manifest['format'], 'tensors': len(artifact_manifest['tensors']), 'bytes': artifact_manifest['files'][0]['bytes'], 'sha256': artifact_manifest['files'][0]['sha256'][:16] + '...'})

reloaded = TAPASTableQAPipeline.from_artifact(artifact_dir, weights_dir=WEIGHTS_DIR, device=pipe.device)
before = [pipe.answer(r['table'], r['question']) for r in test_records[:8]]
after = [reloaded.answer(r['table'], r['question']) for r in test_records[:8]]
parity = {'identical_answers': sum(a['cells'] == b['cells'] and a['aggregation'] == b['aggregation'] and a['aggregation_logits'] == b['aggregation_logits'] for a, b in zip(before, after, strict=True)), 'of': len(before)}
print({'reload_parity': parity, 'reloaded_best_epoch': reloaded.adapter['best_epoch']})
assert parity['identical_answers'] == parity['of']

result_payload = {
    'notebook_source': NOTEBOOK_SOURCE,
    'repository_revision': NOTEBOOK_SOURCE['repository_revision'],
    'model_id': MODEL_ID,
    'model_revision': MODEL_REVISION,
    'model_license': MODEL_LICENSE,
    'snapshot': {'path': str(WEIGHTS_DIR), 'files': len(snapshot['files']), 'total_bytes': snapshot.get('totalBytes'), 'fetched_this_run': fetched, 'weight_file': WEIGHT_FILE, 'weight_format': 'safetensors, digest-verified', 'weight_sha256': pipe.weight_sha256},
    'data_source': data_source,
    'corpus': {'name': CORPUS_NAME, 'repo': CORPUS_REPO, 'revision': CORPUS_REVISION, 'file': CORPUS_FILE, 'bytes': CORPUS_BYTES, 'sha256': CORPUS_SHA256, 'license': CORPUS_LICENSE, 'sample_digest': SAMPLE_DIGEST},
    'inference_contract': {'input_manifest': input_manifest, 'ceilings': ceilings, 'sample': {'name': 'synthetic_cities_table', 'sample_sha256': sample_sha256, 'golds': golds}, 'frozen_report': frozen_scene, 'adapted_report': adapted_scene, 'answers_file': 'outputs/tapas_table_qa_answers.csv'},
    'decision_rule': DECISION_RULE,
    'numeric_answer_source': NUMERIC_ANSWER_SOURCE,
    'comparison': comparison,
    'artifact': {'dir': str(artifact_dir), 'sha256': artifact_manifest['files'][0]['sha256'], 'bytes': artifact_manifest['files'][0]['bytes'], 'tensors': len(artifact_manifest['tensors'])},
    'reload_parity': parity,
    'runtime': {'python': platform.python_version(), 'torch': torch.__version__, 'transformers': transformers.__version__, 'pandas': pandas.__version__, 'device': pipe.device, 'dtype': 'float32'},
}
with open('outputs/tapas_table_qa_result.json', 'w', encoding='utf-8') as handle:
    json.dump(result_payload, handle, indent=2, ensure_ascii=False)
print(sorted(os.listdir('outputs')))

## Interpretation and limits

The frozen WTQ checkpoint is already a strong lookup model on WikiSQL tables — far above the two non-neural baselines, at 0.92 on lookups in the build record — and a bounded fine-tuning of the last two encoder blocks and the heads on 240 questions moves what these questions teach: the operator choice (aggregation accuracy 0.57 → 0.81), a little of the cell selection (0.82 → 0.85) and three questions of denotation accuracy (0.82 → 0.84), with the lookups untouched and a 100 MB adapter that reloads to identical answers. That is the claim: the adaptation contract works end to end on a real labelled table-question set, and the numbers it produces are read on three measures, per question type, against two non-neural baselines and the frozen model rather than in isolation.

The test split is 150 questions from one seeded draw of one sample, the validation split that picks the epoch is 90, denotation accuracy moves in steps of one question, and the build record's own draws show the estimate's fragility: four trainable blocks, or a training draw with as many operator questions as lookups, raised operator accuracy just as much while *costing* lookups, and on another draw denotation accuracy did not move at all. So a gain here says the contract works, not that the adapted model is better on your tables, that a sigmoid threshold or an argmax is a probability, or that the pipeline's arithmetic is right when the cells are wrong — it still returns some cells and some operator for every question, and it is confidently wrong when it is wrong. Fine-tuning on a narrow set can also erode the model elsewhere; the city table re-answered in Section 9 is three questions of evidence about that, not a measurement.

Three things to carry to real data. **Baselines first:** the first-cell floor, the keyword lookup and the frozen model's score on *your* questions are the numbers to read before any adapted one, per question type. **Leakage:** keep every table in one split (the contract splits by normalised table content) and split by source when your tables come from few documents. **Supervision:** the gold cells of a lookup and the number of an operator question are the two shapes the weak-supervision loss understands; a table whose gold cells lie past the 512-token truncation point is refused at training time, and a mislabelled answer is learned without complaint.

Successful execution proves that the recorded repository revision's package, carried in this standalone notebook, can acquire and digest-verify the pinned model snapshot, fetch and digest-verify a real labelled table-question set, validate the demonstrated dataset contract without leakage, execute the inference contract and a bounded fine-tuning, evaluate against two trivial baselines and the frozen model on a table-disjoint split, and emit the shown machine-readable artifacts — without the repository being reachable. It does **not** establish benchmark superiority, denotation accuracy on any other domain or table shape, a usable threshold, or production fitness.

**Optional experiments (they do not affect the default path):** set `TRAINABLE_LAYERS = 4` and compare the artifact size, the operator accuracy and the lookup accuracy; raise `EPOCHS` and watch the validation score pick the epoch while the training loss keeps falling; change `LEARNING_RATE` to `2e-5` and read a smaller, steadier gain; or bring your own JSONL through BYOD and read the two baselines before the adapted number.

## References

- Repository README: https://github.com/kurtvalcorza/tapas-table-question-answering-pipeline/blob/main/README.md
- Repository model card: https://github.com/kurtvalcorza/tapas-table-question-answering-pipeline/blob/main/MODEL_CARD.md
- Weight provenance: https://github.com/kurtvalcorza/tapas-table-question-answering-pipeline/blob/main/docs/WEIGHTS.md
- Upstream model: https://huggingface.co/google/tapas-large-finetuned-wtq
- Upstream code: https://github.com/google-research/tapas
- TAPAS: Weakly Supervised Table Parsing via Pre-training (Herzig et al., 2020): https://arxiv.org/abs/2004.02349
- Understanding tables with intermediate pre-training (Eisenschlos et al., 2020): https://arxiv.org/abs/2010.00571
- Seq2SQL: Generating Structured Queries from Natural Language using Reinforcement Learning (WikiSQL; Zhong, Xiong & Socher, 2017): https://arxiv.org/abs/1709.00103 — dataset https://github.com/salesforce/WikiSQL (BSD-3-Clause)
- Compositional Semantic Parsing on Semi-Structured Tables (WikiTableQuestions): https://arxiv.org/abs/1508.00305
- DIMER Notebook Specification 2.0 and Model Card Specification 1.1 (fleet specs in the ml-worker repository)